# BankSathi · Gemma-2B QLoRA on Colab T4

End-to-end runbook: dataset → pre-GPU audit → QLoRA fine-tune → 5-KPI hold-out gate → INT4 MediaPipe `.task`.

**Before you start**
1. `Runtime → Change runtime type → T4 GPU`.
2. Accept the licence at https://huggingface.co/google/gemma-2-2b-it and create a read token.
3. This notebook already contains all code - just run the cells top to bottom.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Step 1 · Upload code, install, log in (~2 min)

In [ ]:
# Writes the BankSathi pipeline code into /content/gemma_training (no upload needed)
import base64, os
FILES = {
    'banksathi_common.py': 'IiIiU2hhcmVkIGNvbnN0YW50cyBhbmQgaGVscGVycyBmb3IgdGhlIEJhbmtTYXRoaSBHZW1tYS0yQiBwaXBlbGluZS4KCktlcHQgaW4gb25lIHBsYWNlIHNvIHRoZSBkYXRhc2V0IGdlbmVyYXRvciwgdHJhaW5lciBhbmQgZXZhbHVhdG9yIGFncmVlIG9uIHRoZQpleGFjdCBwcm9tcHQgaGVhZGVycywgSlNPTiBjb250cmFjdHMgYW5kIEdlbW1hLTIgY2hhdCB3cmFwcGVyIHVzZWQgYnkKSHlicmlkQWlTZXJ2aWNlLmt0LgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IHJlCmltcG9ydCB1bmljb2RlZGF0YQoKVkVSSUZJRURfREFURSA9ICIyMDI2LTExLTE0IgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBBdXRob3JpdGF0aXZlIHNvdXJjZSBncm91bmRpbmcgbWF0cml4IChndWlkZSBwYWdlIDQpCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tClNPVVJDRVMgPSB7CiAgICAiUkJJX0JDX0JTQkRBXzAxIjogKAogICAgICAgICJDYXNoIHVwIHRvIFJzIDEwLDAwMCBjYW4gYmUgd2l0aGRyYXduIHZpYSBBZVBTIHRodW1iIHNjYW4gYXQgYSBCYW5rIE1pdHJhICIKICAgICAgICAiKEJDIHBvaW50KS4gTm8gZXh0cmEgQkMgZmVlIG9yIGNvbW1pc3Npb24uIENhcnJ5IG9yaWdpbmFsIEFhZGhhYXIgYW5kIHBob25lOyAiCiAgICAgICAgInByaW50ZWQgcmVjZWlwdCBpcyBtYW5kYXRvcnkuIgogICAgKSwKICAgICJSQklfQkNfQUVQU18wNCI6ICgKICAgICAgICAiQW1vdW50cyBhYm92ZSBScyAxMCwwMDAgZXhjZWVkIHRoZSBzdGFuZGFyZCBBZVBTIEJDIGxpbWl0LiBSb3V0ZSB0byB0aGUgYmFuayAiCiAgICAgICAgImJyYW5jaCBvciBBVE0gd2l0aCBwYXNzYm9vaywgQWFkaGFhciBhbmQgUnVQYXkvQVRNIGRlYml0IGNhcmQuIgogICAgKSwKICAgICJSQklfQkNfVFJBTlNGRVJfMDMiOiAoCiAgICAgICAgIkRvbWVzdGljIGNhc2ggZGVwb3NpdCBhbmQgdHJhbnNmZXIgYXJlIHN1cHBvcnRlZCBhdCB0aGUgUG9zdCBPZmZpY2UgKElQUEIpIG9yICIKICAgICAgICAiQmFuayBNaXRyYS4gQ2FycnkgQWFkaGFhciwgcmVjZWl2ZXIgYWNjb3VudCBudW1iZXIgYW5kIElGU0M7IHRha2UgcHJpbnRlZCByZWNlaXB0LiIKICAgICksCiAgICAiUkJJX0JTQkRBX0RCVF8wMiI6ICgKICAgICAgICAiUE1KRFkvQlNCREEgYWNjb3VudHMgbmVlZCB6ZXJvIG1pbmltdW0gYmFsYW5jZSwgYWxsb3cgdW5saW1pdGVkIGRlcG9zaXRzIGFuZCBhdCAiCiAgICAgICAgImxlYXN0IDQgZnJlZSB3aXRoZHJhd2FscyBhIG1vbnRoLiBEQlQgY3JlZGl0cyBjYW4gYmUgY2hlY2tlZCBhdCBCQzsgcGFzc2Jvb2sgaXMgIgogICAgICAgICJwcmludGVkIGF0IHRoZSBicmFuY2guIgogICAgKSwKICAgICJSQklfS0NDX01BU1RFUl8wMSI6ICgKICAgICAgICAiS2lzYW4gQ3JlZGl0IENhcmQgY3JvcCBhbmQgYWxsaWVkIGxvYW5zIHVwIHRvIFJzIDIsMDAsMDAwIGFyZSBjb2xsYXRlcmFsLWZyZWUuICIKICAgICAgICAiQXBwbHkgYXQgdGhlIGJyYW5jaCB3aXRoIEtoYXRhdW5pLCAyIEFhZGhhYXIvUEFOIGNvcGllcywgcGFzc2Jvb2sgYW5kIDMgcGhvdG9zLiIKICAgICksCiAgICAiUkJJX1pFUk9fTElBQklMSVRZXzE5MzAiOiAoCiAgICAgICAgIk9UUC9QSU4vQ1ZWIHJlcXVlc3RzLCBBbnlEZXNrIG9yIC5hcGsgaW5zdGFsbHMsIGJpdC5seSBLWUMgbGlua3MsIG9yIEFlUFMgIgogICAgICAgICJkb3VibGUtdGh1bWIgd2l0aG91dCByZWNlaXB0IGFyZSBmcmF1ZC4gQ2FsbCAxOTMwIGltbWVkaWF0ZWx5IGFuZCBub3RpZnkgdGhlICIKICAgICAgICAiYmFuayB3aXRoaW4gMyB3b3JraW5nIGRheXMgZm9yIHplcm8gbGlhYmlsaXR5LiIKICAgICksCn0KCgpkZWYgc291cmNlX3NuaXBwZXQoc291cmNlX2lkOiBzdHIpIC0+IHN0cjoKICAgIHJldHVybiBmIlt7c291cmNlX2lkfSB8IFZlcmlmaWVkIHtWRVJJRklFRF9EQVRFfV0ge1NPVVJDRVNbc291cmNlX2lkXX0iCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBQcm9tcHQgY29udHJhY3RzIChndWlkZSBwYWdlIDQpIC0gbXVzdCBtYXRjaCBIeWJyaWRBaVNlcnZpY2Uua3QgYnl0ZS1mb3ItYnl0ZQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQpUQVNLX0FfSEVBREVSID0gKAogICAgIllvdSBhcmUgQmFua1NhdGhpLCBhbiBvZmZsaW5lLWZpcnN0IHJ1cmFsIEluZGlhbiBiYW5raW5nIGFzc2lzdGFudC4gUmVwaHJhc2UgT05MWSAiCiAgICAiZnJvbSBWZXJpZmllZCBDb250ZXh0LiBJZiB1bnN1cmUsIHNldCBhYnN0YWluZWQ9dHJ1ZSBhbmQgdGFzaz1cIk5PTkVcIi5cbiIKICAgICJDb3JlIFJ1bGVzOiBDYXNoIDw9IDEwMDAwIC0+IEJDX1BPSU5UOyA+IDEwMDAwIC0+IEJSQU5DSC9BVE07IEtDQyA8PSAyTCAtPiBCUkFOQ0g7ICIKICAgICJUcmFuc2ZlciAtPiBJUFBCL0JDOyBPVFAvS1lDIC0+IDE5MzAuXG4iCiAgICAiUmV0dXJuIEpTT04gT05MWSB3aXRoIGtleXM6IHJlcGx5RW4sIHJlcGx5SGksIHRhc2ssIHN1Z2dlc3RlZEFtb3VudEluciwgIgogICAgImlzU2NhbUFsZXJ0LCBhYnN0YWluZWQuIgopCgpUQVNLX0JfSEVBREVSID0gKAogICAgIkFuYWx5emUgdGhpcyBiYW5raW5nIFNNUyBvciBjYWxsIHNjZW5hcmlvIGZvciBydXJhbCBmcmF1ZCBpbiBJbmRpYS4gQWR2aXNlIGNhbGxpbmcgIgogICAgIjE5MzAgYW5kIHJlcG9ydGluZyB3aXRoaW4gMyBkYXlzIG9uIEhJR0hfUklTSy5cbiIKICAgICJSZXR1cm4gSlNPTiBPTkxZIHdpdGgga2V5czogcmlza0xldmVsLCBleHBsYW5hdGlvbkVuLCBleHBsYW5hdGlvbkhpLCBhY3Rpb25UaXBFbiwgIgogICAgImFjdGlvblRpcEhpLiIKKQoKTk9fQ09OVEVYVCA9ICJub25lICh1c2UgQ29yZSBSdWxlcykiCgoKZGVmIGJ1aWxkX3Rhc2tfYV9wcm9tcHQocXVlcnk6IHN0ciwgY29udGV4dF9pZHM6IGxpc3Rbc3RyXSkgLT4gc3RyOgogICAgY3R4ID0gIlxuIi5qb2luKHNvdXJjZV9zbmlwcGV0KHMpIGZvciBzIGluIGNvbnRleHRfaWRzKSBpZiBjb250ZXh0X2lkcyBlbHNlIE5PX0NPTlRFWFQKICAgIHJldHVybiBmIntUQVNLX0FfSEVBREVSfVxuVmVyaWZpZWQgQ29udGV4dDpcbntjdHh9XG5Vc2VyOiB7cXVlcnl9IgoKCmRlZiBidWlsZF90YXNrX2JfcHJvbXB0KG1lc3NhZ2U6IHN0cikgLT4gc3RyOgogICAgcmV0dXJuIGYie1RBU0tfQl9IRUFERVJ9XG5NZXNzYWdlOiB7bWVzc2FnZX0iCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBHZW1tYS0yIGNoYXQgd3JhcHBlciAoZ3VpZGUgcGFnZSA0KQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQpVU0VSX1RVUk4gPSAiPHN0YXJ0X29mX3R1cm4+dXNlclxuIgpNT0RFTF9UVVJOID0gIjxzdGFydF9vZl90dXJuPm1vZGVsXG4iCkVORF9UVVJOID0gIjxlbmRfb2ZfdHVybj4iCgoKZGVmIHdyYXBfcHJvbXB0KHByb21wdDogc3RyKSAtPiBzdHI6CiAgICAiIiJFdmVyeXRoaW5nIHVwIHRvIGFuZCBpbmNsdWRpbmcgdGhlIHRlbXBsYXRlIGJvdW5kYXJ5IChsb3NzLW1hc2tlZCkuIiIiCiAgICByZXR1cm4gZiJ7VVNFUl9UVVJOfXtwcm9tcHR9e0VORF9UVVJOfVxue01PREVMX1RVUk59IgoKCmRlZiB3cmFwX2Z1bGwocHJvbXB0OiBzdHIsIGNvbXBsZXRpb246IHN0cikgLT4gc3RyOgogICAgcmV0dXJuIGYie3dyYXBfcHJvbXB0KHByb21wdCl9e2NvbXBsZXRpb259e0VORF9UVVJOfSIKCgpkZWYgY29tcGFjdF9qc29uKG9iajogZGljdCkgLT4gc3RyOgogICAgcmV0dXJuIGpzb24uZHVtcHMob2JqLCBlbnN1cmVfYXNjaWk9RmFsc2UsIHNlcGFyYXRvcnM9KCIsIiwgIjoiKSkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIEpTT04gc2NoZW1hcwojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQpUQVNLUyA9ICgiV0lUSERSQVciLCAiVFJBTlNGRVIiLCAiUEFTU0JPT0tfREJUIiwgIkxPQU5fS0NDIiwgIk5PTkUiKQpSSVNLX0xFVkVMUyA9ICgiU0FGRSIsICJDQVVUSU9OIiwgIkhJR0hfUklTSyIpCgpUQVNLX0FfS0VZUyA9ICgicmVwbHlFbiIsICJyZXBseUhpIiwgInRhc2siLCAic3VnZ2VzdGVkQW1vdW50SW5yIiwgImlzU2NhbUFsZXJ0IiwgImFic3RhaW5lZCIpClRBU0tfQl9LRVlTID0gKCJyaXNrTGV2ZWwiLCAiZXhwbGFuYXRpb25FbiIsICJleHBsYW5hdGlvbkhpIiwgImFjdGlvblRpcEVuIiwgImFjdGlvblRpcEhpIikKClJFUExZX1dPUkRfQ0FQID0gNDUKRVhQTEFOQVRJT05fV09SRF9DQVAgPSA1MAoKCmRlZiB3b3JkX2NvdW50KHRleHQ6IHN0cikgLT4gaW50OgogICAgcmV0dXJuIGxlbih0ZXh0LnNwbGl0KCkpCgoKREVWQU5BR0FSSV9SRSA9IHJlLmNvbXBpbGUociJb4KSALeClv10iKQpMQVRJTl9SRSA9IHJlLmNvbXBpbGUociJbQS1aYS16XSIpCgoKZGVmIGRldmFuYWdhcmlfb2sodGV4dDogc3RyLCBtaW5fcmF0aW86IGZsb2F0ID0gMC44NSkgLT4gYm9vbDoKICAgICIiIlRydWUgd2hlbiB0ZXh0IGlzIHByZWRvbWluYW50bHkgRGV2YW5hZ2FyaSAoYWxsb3dzIGRpZ2l0cywgUnMgc3ltYm9sLCBhIGZldyBhY3JvbnltcykuIiIiCiAgICBkZXYgPSBsZW4oREVWQU5BR0FSSV9SRS5maW5kYWxsKHRleHQpKQogICAgbGF0ID0gbGVuKExBVElOX1JFLmZpbmRhbGwodGV4dCkpCiAgICByZXR1cm4gZGV2ID4gMCBhbmQgZGV2IC8gKGRldiArIGxhdCkgPj0gbWluX3JhdGlvCgoKZGVmIHZhbGlkYXRlX3Rhc2tfYShvYmosIHN0cmljdF9jYXBzOiBib29sID0gVHJ1ZSkgLT4gbGlzdFtzdHJdOgogICAgZXJycyA9IFtdCiAgICBpZiBub3QgaXNpbnN0YW5jZShvYmosIGRpY3QpOgogICAgICAgIHJldHVybiBbIm5vdCBhIEpTT04gb2JqZWN0Il0KICAgIGlmIHNldChvYmoua2V5cygpKSAhPSBzZXQoVEFTS19BX0tFWVMpOgogICAgICAgIGVycnMuYXBwZW5kKGYia2V5cyBtaXNtYXRjaDoge3NvcnRlZChvYmoua2V5cygpKX0iKQogICAgICAgIHJldHVybiBlcnJzCiAgICBpZiBub3QgaXNpbnN0YW5jZShvYmpbInJlcGx5RW4iXSwgc3RyKSBvciBub3Qgb2JqWyJyZXBseUVuIl0uc3RyaXAoKToKICAgICAgICBlcnJzLmFwcGVuZCgicmVwbHlFbiBlbXB0eSIpCiAgICBpZiBub3QgaXNpbnN0YW5jZShvYmpbInJlcGx5SGkiXSwgc3RyKSBvciBub3Qgb2JqWyJyZXBseUhpIl0uc3RyaXAoKToKICAgICAgICBlcnJzLmFwcGVuZCgicmVwbHlIaSBlbXB0eSIpCiAgICBpZiBvYmpbInRhc2siXSBub3QgaW4gVEFTS1M6CiAgICAgICAgZXJycy5hcHBlbmQoZiJiYWQgdGFzayB7b2JqWyd0YXNrJ10hcn0iKQogICAgYW10ID0gb2JqWyJzdWdnZXN0ZWRBbW91bnRJbnIiXQogICAgaWYgaXNpbnN0YW5jZShhbXQsIGJvb2wpIG9yIG5vdCBpc2luc3RhbmNlKGFtdCwgaW50KSBvciBhbXQgPCAwOgogICAgICAgIGVycnMuYXBwZW5kKCJzdWdnZXN0ZWRBbW91bnRJbnIgbXVzdCBiZSBub24tbmVnYXRpdmUgaW50IikKICAgIGZvciBrIGluICgiaXNTY2FtQWxlcnQiLCAiYWJzdGFpbmVkIik6CiAgICAgICAgaWYgbm90IGlzaW5zdGFuY2Uob2JqW2tdLCBib29sKToKICAgICAgICAgICAgZXJycy5hcHBlbmQoZiJ7a30gbXVzdCBiZSBib29sIikKICAgIGlmIG9iai5nZXQoImFic3RhaW5lZCIpIGlzIFRydWUgYW5kIG9iai5nZXQoInRhc2siKSAhPSAiTk9ORSI6CiAgICAgICAgZXJycy5hcHBlbmQoImFic3RhaW5lZD10cnVlIHJlcXVpcmVzIHRhc2s9Tk9ORSIpCiAgICBpZiBzdHJpY3RfY2FwcyBhbmQgbm90IGVycnM6CiAgICAgICAgaWYgd29yZF9jb3VudChvYmpbInJlcGx5RW4iXSkgPiBSRVBMWV9XT1JEX0NBUDoKICAgICAgICAgICAgZXJycy5hcHBlbmQoInJlcGx5RW4gPiA0NSB3b3JkcyIpCiAgICAgICAgaWYgd29yZF9jb3VudChvYmpbInJlcGx5SGkiXSkgPiBSRVBMWV9XT1JEX0NBUDoKICAgICAgICAgICAgZXJycy5hcHBlbmQoInJlcGx5SGkgPiA0NSB3b3JkcyIpCiAgICByZXR1cm4gZXJycwoKCmRlZiB2YWxpZGF0ZV90YXNrX2Iob2JqLCBzdHJpY3RfY2FwczogYm9vbCA9IFRydWUpIC0+IGxpc3Rbc3RyXToKICAgIGVycnMgPSBbXQogICAgaWYgbm90IGlzaW5zdGFuY2Uob2JqLCBkaWN0KToKICAgICAgICByZXR1cm4gWyJub3QgYSBKU09OIG9iamVjdCJdCiAgICBpZiBzZXQob2JqLmtleXMoKSkgIT0gc2V0KFRBU0tfQl9LRVlTKToKICAgICAgICByZXR1cm4gW2Yia2V5cyBtaXNtYXRjaDoge3NvcnRlZChvYmoua2V5cygpKX0iXQogICAgaWYgb2JqWyJyaXNrTGV2ZWwiXSBub3QgaW4gUklTS19MRVZFTFM6CiAgICAgICAgZXJycy5hcHBlbmQoZiJiYWQgcmlza0xldmVsIHtvYmpbJ3Jpc2tMZXZlbCddIXJ9IikKICAgIGZvciBrIGluIFRBU0tfQl9LRVlTWzE6XToKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShvYmpba10sIHN0cikgb3Igbm90IG9ialtrXS5zdHJpcCgpOgogICAgICAgICAgICBlcnJzLmFwcGVuZChmIntrfSBlbXB0eSIpCiAgICBpZiBzdHJpY3RfY2FwcyBhbmQgbm90IGVycnM6CiAgICAgICAgZm9yIGsgaW4gVEFTS19CX0tFWVNbMTpdOgogICAgICAgICAgICBpZiB3b3JkX2NvdW50KG9ialtrXSkgPiBFWFBMQU5BVElPTl9XT1JEX0NBUDoKICAgICAgICAgICAgICAgIGVycnMuYXBwZW5kKGYie2t9ID4gNTAgd29yZHMiKQogICAgcmV0dXJuIGVycnMKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIERlZHVwZSAvIGxlYWthZ2UgaGFzaGluZyAoZ3VpZGUgcGFnZSAzLCBydWxlIDMpCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCmRlZiBub3JtYWxpemVfcXVlcnkodGV4dDogc3RyKSAtPiBzdHI6CiAgICB0ZXh0ID0gdW5pY29kZWRhdGEubm9ybWFsaXplKCJORkMiLCB0ZXh0KS5sb3dlcigpCiAgICB0ZXh0ID0gIiIuam9pbihjaCBmb3IgY2ggaW4gdGV4dCBpZiBub3QgdW5pY29kZWRhdGEuY2F0ZWdvcnkoY2gpLnN0YXJ0c3dpdGgoKCJQIiwgIlMiKSkpCiAgICByZXR1cm4gIiAiLmpvaW4odGV4dC5zcGxpdCgpKQoKCmRlZiBxdWVyeV9oYXNoKHRleHQ6IHN0cikgLT4gc3RyOgogICAgcmV0dXJuIGhhc2hsaWIuc2hhMjU2KG5vcm1hbGl6ZV9xdWVyeSh0ZXh0KS5lbmNvZGUoInV0Zi04IikpLmhleGRpZ2VzdCgpCgoKZGVmIHJlYWRfanNvbmwocGF0aDogc3RyKSAtPiBsaXN0W2RpY3RdOgogICAgcm93cyA9IFtdCiAgICB3aXRoIG9wZW4ocGF0aCwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICBmb3IgbGluZSBpbiBmOgogICAgICAgICAgICBsaW5lID0gbGluZS5zdHJpcCgpCiAgICAgICAgICAgIGlmIGxpbmU6CiAgICAgICAgICAgICAgICByb3dzLmFwcGVuZChqc29uLmxvYWRzKGxpbmUpKQogICAgcmV0dXJuIHJvd3MKCgpkZWYgd3JpdGVfanNvbmwocGF0aDogc3RyLCByb3dzOiBsaXN0W2RpY3RdKSAtPiBOb25lOgogICAgd2l0aCBvcGVuKHBhdGgsICJ3IiwgZW5jb2Rpbmc9InV0Zi04IiwgbmV3bGluZT0iXG4iKSBhcyBmOgogICAgICAgIGZvciByIGluIHJvd3M6CiAgICAgICAgICAgIGYud3JpdGUoanNvbi5kdW1wcyhyLCBlbnN1cmVfYXNjaWk9RmFsc2UpICsgIlxuIikK',
    'dataset_generator.py': '...',
    'eval_gemma_banksathi.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJCYW5rU2F0aGkgNS1LUEkgcmVsZWFzZSBnYXRlIChndWlkZSBwYWdlcyAxLCAzLCA3KS4KClRocmVlIG1vZGVzOgoKMS4gUHJlLUdQVSBkYXRhc2V0IGF1ZGl0IChubyBtb2RlbCk6CiAgICAgcHl0aG9uMyBldmFsX2dlbW1hX2JhbmtzYXRoaS5weSAtLXRyYWluLWZpbGUgZGF0YS90cmFpbi5qc29ubCAtLWV2YWwtZmlsZSBkYXRhL2V2YWwuanNvbmwKICAgQ2hlY2tzIHZhbGlkIEpTT04sIHNjaGVtYSwgPD00NS13b3JkIFRUUyBjYXBzLCBEZXZhbmFnYXJpIHBhcml0eSwgU0hBLTI1NiB0cmFpbi9ldmFsIGxlYWthZ2UsCiAgIGNsYXNzIGJhbGFuY2UuIFJlcXVpcmVzIHZhbGlkX2pzb25fcGN0ID09IDEwMCBhbmQgZGV2YW5hZ2FyaV9wYXJpdHlfcGN0ID09IDEwMC4KCjIuIExpdmUgbW9kZWwgYmVuY2htYXJrIG9uIHRoZSBob2xkLW91dCBzZXQ6CiAgICAgcHl0aG9uMyBldmFsX2dlbW1hX2JhbmtzYXRoaS5weSAtLW1vZGVsLWRpciBvdXRwdXRzL2dlbW1hMi0yYi1iYW5rc2F0aGkvbWVyZ2VkX2ZwMTYgLS1ldmFsLWZpbGUgZGF0YS9ldmFsLmpzb25sCiAgIEdyZWVkeS1kZWNvZGVzIGV2ZXJ5IGV2YWwgcHJvbXB0IGFuZCBzY29yZXMgdGhlIDUgS1BJczoKICAgICB2YWxpZF9qc29uX3BjdCA+PSA5OSB8IGludGVudF9hY2N1cmFjeV9wY3QgPj0gOTYgfCBzY2FtZ3VhcmRfcmVjYWxsX3BjdCA9PSAxMDAKICAgICBzYWZlX2Fic3RlbnRpb25fcGN0ID49IDk1IHwgYmlsaW5ndWFsX3R0c19wY3QgPj0gOTgKCjMuIFNjb3JlIHNhdmVkIHJhdyBvdXRwdXRzIChlLmcuIGZyb20gdGhlIHBob25lIG9yIGEgcHJldmlvdXMgcnVuKToKICAgICBweXRob24zIGV2YWxfZ2VtbWFfYmFua3NhdGhpLnB5IC0tZXZhbC1maWxlIGRhdGEvZXZhbC5qc29ubCAtLXByZWRpY3Rpb25zLWZpbGUgcHJlZHMuanNvbmwKICAgcHJlZHMuanNvbmw6IG9uZSB7ImlkIjogLi4uLCAib3V0cHV0IjogIjxyYXcgbW9kZWwgdGV4dD4ifSBwZXIgbGluZS4KCkV4aXQgY29kZSBpcyAwIG9ubHkgd2hlbiBldmVyeSBnYXRlIHBhc3Nlcy4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgY29sbGVjdGlvbnMKaW1wb3J0IGpzb24KaW1wb3J0IHN5cwoKZnJvbSBiYW5rc2F0aGlfY29tbW9uIGltcG9ydCAoCiAgICBFTkRfVFVSTiwKICAgIFJFUExZX1dPUkRfQ0FQLAogICAgRVhQTEFOQVRJT05fV09SRF9DQVAsCiAgICBkZXZhbmFnYXJpX29rLAogICAgcmVhZF9qc29ubCwKICAgIHZhbGlkYXRlX3Rhc2tfYSwKICAgIHZhbGlkYXRlX3Rhc2tfYiwKICAgIHdvcmRfY291bnQsCiAgICB3cmFwX3Byb21wdCwKKQoKS1BJX0dBVEVTID0gewogICAgInZhbGlkX2pzb25fcGN0IjogOTkuMCwKICAgICJpbnRlbnRfYWNjdXJhY3lfcGN0IjogOTYuMCwKICAgICJzY2FtZ3VhcmRfcmVjYWxsX3BjdCI6IDEwMC4wLAogICAgInNhZmVfYWJzdGVudGlvbl9wY3QiOiA5NS4wLAogICAgImJpbGluZ3VhbF90dHNfcGN0IjogOTguMCwKfQoKCmRlZiBwY3QobnVtOiBpbnQsIGRlbjogaW50KSAtPiBmbG9hdDoKICAgIHJldHVybiByb3VuZCgxMDAuMCAqIG51bSAvIGRlbiwgMikgaWYgZGVuIGVsc2UgMTAwLjAKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIE1vZGUgMTogZGF0YXNldCBhdWRpdAojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQpkZWYgYXVkaXRfZGF0YXNldCh0cmFpbjogbGlzdFtkaWN0XSwgZXZhbHM6IGxpc3RbZGljdF0pIC0+IGRpY3Q6CiAgICByb3dzID0gdHJhaW4gKyBldmFscwogICAgdmFsaWRfanNvbiA9IHBhcml0eSA9IGNhcHMgPSAwCiAgICBwcm9ibGVtcyA9IFtdCiAgICBmb3IgciBpbiByb3dzOgogICAgICAgIHRyeToKICAgICAgICAgICAgb2JqID0ganNvbi5sb2FkcyhyWyJjb21wbGV0aW9uIl0pCiAgICAgICAgZXhjZXB0IGpzb24uSlNPTkRlY29kZUVycm9yIGFzIGU6CiAgICAgICAgICAgIHByb2JsZW1zLmFwcGVuZCgoci5nZXQoImlkIiksIGYianNvbjoge2V9IikpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgdmFsaWRhdG9yID0gdmFsaWRhdGVfdGFza19hIGlmIHJbInRhc2tfdHlwZSJdID09ICJhc3Npc3RhbnQiIGVsc2UgdmFsaWRhdGVfdGFza19iCiAgICAgICAgc2NoZW1hX2VycnMgPSB2YWxpZGF0b3Iob2JqLCBzdHJpY3RfY2Fwcz1GYWxzZSkKICAgICAgICBpZiBzY2hlbWFfZXJyczoKICAgICAgICAgICAgcHJvYmxlbXMuYXBwZW5kKChyLmdldCgiaWQiKSwgZiJzY2hlbWE6IHtzY2hlbWFfZXJyc30iKSkKICAgICAgICAgICAgY29udGludWUKICAgICAgICB2YWxpZF9qc29uICs9IDEKICAgICAgICBpZiByWyJ0YXNrX3R5cGUiXSA9PSAiYXNzaXN0YW50IjoKICAgICAgICAgICAgaGlfb2sgPSBkZXZhbmFnYXJpX29rKG9ialsicmVwbHlIaSJdKQogICAgICAgICAgICBjYXBfb2sgPSB3b3JkX2NvdW50KG9ialsicmVwbHlFbiJdKSA8PSBSRVBMWV9XT1JEX0NBUCBhbmQgd29yZF9jb3VudChvYmpbInJlcGx5SGkiXSkgPD0gUkVQTFlfV09SRF9DQVAKICAgICAgICBlbHNlOgogICAgICAgICAgICBoaV9vayA9IGRldmFuYWdhcmlfb2sob2JqWyJleHBsYW5hdGlvbkhpIl0pIGFuZCBkZXZhbmFnYXJpX29rKG9ialsiYWN0aW9uVGlwSGkiXSkKICAgICAgICAgICAgY2FwX29rID0gYWxsKHdvcmRfY291bnQob2JqW2tdKSA8PSBFWFBMQU5BVElPTl9XT1JEX0NBUAogICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGsgaW4gKCJleHBsYW5hdGlvbkVuIiwgImV4cGxhbmF0aW9uSGkiLCAiYWN0aW9uVGlwRW4iLCAiYWN0aW9uVGlwSGkiKSkKICAgICAgICBwYXJpdHkgKz0gaGlfb2sKICAgICAgICBjYXBzICs9IGNhcF9vawogICAgICAgIGlmIG5vdCBoaV9vazoKICAgICAgICAgICAgcHJvYmxlbXMuYXBwZW5kKChyLmdldCgiaWQiKSwgIkRldmFuYWdhcmkgcGFyaXR5IikpCiAgICAgICAgaWYgbm90IGNhcF9vazoKICAgICAgICAgICAgcHJvYmxlbXMuYXBwZW5kKChyLmdldCgiaWQiKSwgIndvcmQgY2FwIikpCiAgICAgICAgaWYgbm90IHJbInRleHQiXS5lbmRzd2l0aChyWyJjb21wbGV0aW9uIl0gKyBFTkRfVFVSTikgb3Igbm90IHJbInRleHQiXS5zdGFydHN3aXRoKHdyYXBfcHJvbXB0KHJbInByb21wdCJdKSk6CiAgICAgICAgICAgIHByb2JsZW1zLmFwcGVuZCgoci5nZXQoImlkIiksICJjaGF0IHdyYXBwZXIgbWlzbWF0Y2giKSkKCiAgICB0cmFpbl9oID0ge3JbInF1ZXJ5X3NoYTI1NiJdIGZvciByIGluIHRyYWlufQogICAgZXZhbF9oID0ge3JbInF1ZXJ5X3NoYTI1NiJdIGZvciByIGluIGV2YWxzfQogICAgZHVwX3dpdGhpbiA9IGxlbihyb3dzKSAtIGxlbih7clsicXVlcnlfc2hhMjU2Il0gZm9yIHIgaW4gcm93c30pCiAgICBidWNrZXRzID0gY29sbGVjdGlvbnMuQ291bnRlcihyWyJidWNrZXQiXSBmb3IgciBpbiByb3dzKQogICAgdG90YWwgPSBsZW4ocm93cykKICAgIHdpdGhkcmF3X3NoYXJlID0gcGN0KHN1bSh2IGZvciBrLCB2IGluIGJ1Y2tldHMuaXRlbXMoKSBpZiBrLnN0YXJ0c3dpdGgoIndpdGhkcmF3XyIpKSwgdG90YWwpCiAgICBtYXhfYnVja2V0X3NoYXJlID0gbWF4KHBjdCh2LCB0b3RhbCkgZm9yIHYgaW4gYnVja2V0cy52YWx1ZXMoKSkgaWYgYnVja2V0cyBlbHNlIDAuMAoKICAgIHJlcG9ydCA9IHsKICAgICAgICAibW9kZSI6ICJkYXRhc2V0X2F1ZGl0IiwKICAgICAgICAidG90YWxfcm93cyI6IHRvdGFsLAogICAgICAgICJ0cmFpbl9yb3dzIjogbGVuKHRyYWluKSwKICAgICAgICAiZXZhbF9yb3dzIjogbGVuKGV2YWxzKSwKICAgICAgICAidmFsaWRfanNvbl9wY3QiOiBwY3QodmFsaWRfanNvbiwgdG90YWwpLAogICAgICAgICJkZXZhbmFnYXJpX3Bhcml0eV9wY3QiOiBwY3QocGFyaXR5LCB0b3RhbCksCiAgICAgICAgInR0c193b3JkX2NhcF9wY3QiOiBwY3QoY2FwcywgdG90YWwpLAogICAgICAgICJ0cmFpbl9ldmFsX2xlYWthZ2VfY291bnQiOiBsZW4odHJhaW5faCAmIGV2YWxfaCksCiAgICAgICAgImR1cGxpY2F0ZV9xdWVyeV9jb3VudCI6IGR1cF93aXRoaW4sCiAgICAgICAgIndpdGhkcmF3X3NoYXJlX3BjdCI6IHdpdGhkcmF3X3NoYXJlLAogICAgICAgICJtYXhfYnVja2V0X3NoYXJlX3BjdCI6IG1heF9idWNrZXRfc2hhcmUsCiAgICAgICAgImJ1Y2tldHMiOiBkaWN0KGJ1Y2tldHMubW9zdF9jb21tb24oKSksCiAgICAgICAgInJlZ2lzdGVycyI6IGRpY3QoY29sbGVjdGlvbnMuQ291bnRlcihyWyJyZWdpc3RlciJdIGZvciByIGluIHJvd3MpLm1vc3RfY29tbW9uKCkpLAogICAgICAgICJwcm9ibGVtc19zYW1wbGUiOiBwcm9ibGVtc1s6MjBdLAogICAgfQogICAgcmVwb3J0WyJnYXRlcyJdID0gewogICAgICAgICJ2YWxpZF9qc29uX3BjdCA9PSAxMDAiOiByZXBvcnRbInZhbGlkX2pzb25fcGN0Il0gPT0gMTAwLjAsCiAgICAgICAgImRldmFuYWdhcmlfcGFyaXR5X3BjdCA9PSAxMDAiOiByZXBvcnRbImRldmFuYWdhcmlfcGFyaXR5X3BjdCJdID09IDEwMC4wLAogICAgICAgICJ0dHNfd29yZF9jYXBfcGN0ID09IDEwMCI6IHJlcG9ydFsidHRzX3dvcmRfY2FwX3BjdCJdID09IDEwMC4wLAogICAgICAgICJ0cmFpbl9ldmFsX2xlYWthZ2VfY291bnQgPT0gMCI6IHJlcG9ydFsidHJhaW5fZXZhbF9sZWFrYWdlX2NvdW50Il0gPT0gMCwKICAgICAgICAiZHVwbGljYXRlX3F1ZXJ5X2NvdW50ID09IDAiOiBkdXBfd2l0aGluID09IDAsCiAgICAgICAgIndpdGhkcmF3X3NoYXJlX3BjdCA8PSAyNSI6IHdpdGhkcmF3X3NoYXJlIDw9IDI1LjAsCiAgICAgICAgIm1heF9idWNrZXRfc2hhcmVfcGN0IDw9IDE2LjUiOiBtYXhfYnVja2V0X3NoYXJlIDw9IDE2LjUsCiAgICAgICAgIm5vIHdyYXBwZXIgcHJvYmxlbXMiOiBub3QgYW55KHBbMV0gPT0gImNoYXQgd3JhcHBlciBtaXNtYXRjaCIgZm9yIHAgaW4gcHJvYmxlbXMpLAogICAgfQogICAgcmVwb3J0WyJwYXNzZWQiXSA9IGFsbChyZXBvcnRbImdhdGVzIl0udmFsdWVzKCkpCiAgICByZXR1cm4gcmVwb3J0CgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBNb2RlIDIvMzogS1BJIHNjb3Jpbmcgb2YgcmF3IG1vZGVsIG91dHB1dHMKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KZGVmIHN0cmljdF9wYXJzZShyYXc6IHN0cik6CiAgICAiIiJTdHJpY3Q6IG9ubHkgc3Vycm91bmRpbmcgd2hpdGVzcGFjZSAvIGEgdHJhaWxpbmcgPGVuZF9vZl90dXJuPiBpcyB0b2xlcmF0ZWQgKG5vIGZlbmNlIHN0cmlwcGluZykuIiIiCiAgICB0ZXh0ID0gcmF3LnN0cmlwKCkKICAgIGlmIHRleHQuZW5kc3dpdGgoRU5EX1RVUk4pOgogICAgICAgIHRleHQgPSB0ZXh0WzogLWxlbihFTkRfVFVSTildLnN0cmlwKCkKICAgIHRyeToKICAgICAgICByZXR1cm4ganNvbi5sb2Fkcyh0ZXh0KQogICAgZXhjZXB0IChqc29uLkpTT05EZWNvZGVFcnJvciwgVmFsdWVFcnJvcik6CiAgICAgICAgcmV0dXJuIE5vbmUKCgpkZWYgc2NvcmVfb3V0cHV0cyhldmFsczogbGlzdFtkaWN0XSwgb3V0cHV0czogbGlzdFtzdHJdKSAtPiBkaWN0OgogICAgYXNzZXJ0IGxlbihldmFscykgPT0gbGVuKG91dHB1dHMpLCAib25lIG91dHB1dCBwZXIgZXZhbCByb3cgcmVxdWlyZWQiCiAgICBuX3ZhbGlkID0gMAogICAgaW50ZW50X2hpdCA9IGludGVudF9uID0gMAogICAgc2NhbV9oaXQgPSBzY2FtX24gPSAwCiAgICBhYnN0X2hpdCA9IGFic3RfbiA9IDAKICAgIHR0c19oaXQgPSB0dHNfbiA9IDAKICAgIG92ZXJfYWJzdGFpbiA9IGFuc3dlcmFibGVfbiA9IDAKICAgIHNhZmVfZmFsc2VfYWxhcm0gPSBzYWZlX24gPSAwCiAgICBwZXJfYnVja2V0ID0gY29sbGVjdGlvbnMuZGVmYXVsdGRpY3QobGFtYmRhOiBbMCwgMF0pCiAgICBmYWlsdXJlcyA9IFtdCgogICAgZm9yIHJvdywgcmF3IGluIHppcChldmFscywgb3V0cHV0cyk6CiAgICAgICAgZ29sZCA9IGpzb24ubG9hZHMocm93WyJjb21wbGV0aW9uIl0pCiAgICAgICAgcHJlZCA9IHN0cmljdF9wYXJzZShyYXcpCiAgICAgICAgaXNfYSA9IHJvd1sidGFza190eXBlIl0gPT0gImFzc2lzdGFudCIKICAgICAgICB2YWxpZGF0b3IgPSB2YWxpZGF0ZV90YXNrX2EgaWYgaXNfYSBlbHNlIHZhbGlkYXRlX3Rhc2tfYgogICAgICAgIHZhbGlkID0gcHJlZCBpcyBub3QgTm9uZSBhbmQgbm90IHZhbGlkYXRvcihwcmVkLCBzdHJpY3RfY2Fwcz1GYWxzZSkKICAgICAgICBuX3ZhbGlkICs9IHZhbGlkCiAgICAgICAgb2sgPSBGYWxzZQoKICAgICAgICBpZiBpc19hOgogICAgICAgICAgICBpbnRlbnRfbiArPSAxCiAgICAgICAgICAgIGludGVudF9vayA9IHZhbGlkIGFuZCBwcmVkWyJ0YXNrIl0gPT0gZ29sZFsidGFzayJdCiAgICAgICAgICAgIGludGVudF9oaXQgKz0gaW50ZW50X29rCiAgICAgICAgICAgIGlmIHZhbGlkOgogICAgICAgICAgICAgICAgdHRzX24gKz0gMQogICAgICAgICAgICAgICAgdHRzX29rID0gKGRldmFuYWdhcmlfb2socHJlZFsicmVwbHlIaSJdKSBhbmQgd29yZF9jb3VudChwcmVkWyJyZXBseUVuIl0pIDw9IFJFUExZX1dPUkRfQ0FQCiAgICAgICAgICAgICAgICAgICAgICAgICAgYW5kIHdvcmRfY291bnQocHJlZFsicmVwbHlIaSJdKSA8PSBSRVBMWV9XT1JEX0NBUCkKICAgICAgICAgICAgICAgIHR0c19oaXQgKz0gdHRzX29rCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICB0dHNfbiArPSAxCiAgICAgICAgICAgIGlmIGdvbGRbImlzU2NhbUFsZXJ0Il06CiAgICAgICAgICAgICAgICBzY2FtX24gKz0gMQogICAgICAgICAgICAgICAgc19vayA9IHZhbGlkIGFuZCBwcmVkWyJpc1NjYW1BbGVydCJdIGlzIFRydWUgYW5kICIxOTMwIiBpbiBwcmVkWyJyZXBseUVuIl0KICAgICAgICAgICAgICAgIHNjYW1faGl0ICs9IHNfb2sKICAgICAgICAgICAgICAgIG9rID0gc19vawogICAgICAgICAgICBlbGlmIGdvbGRbImFic3RhaW5lZCJdOgogICAgICAgICAgICAgICAgYWJzdF9uICs9IDEKICAgICAgICAgICAgICAgIGFfb2sgPSB2YWxpZCBhbmQgcHJlZFsiYWJzdGFpbmVkIl0gaXMgVHJ1ZSBhbmQgcHJlZFsidGFzayJdID09ICJOT05FIgogICAgICAgICAgICAgICAgYWJzdF9oaXQgKz0gYV9vawogICAgICAgICAgICAgICAgb2sgPSBhX29rCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBhbnN3ZXJhYmxlX24gKz0gMQogICAgICAgICAgICAgICAgaWYgdmFsaWQgYW5kIHByZWRbImFic3RhaW5lZCJdOgogICAgICAgICAgICAgICAgICAgIG92ZXJfYWJzdGFpbiArPSAxCiAgICAgICAgICAgICAgICBvayA9IGludGVudF9vayBhbmQgbm90IHByZWRbImFic3RhaW5lZCJdCiAgICAgICAgZWxzZToKICAgICAgICAgICAgdHRzX24gKz0gMQogICAgICAgICAgICBpZiB2YWxpZDoKICAgICAgICAgICAgICAgIHR0c19oaXQgKz0gKGRldmFuYWdhcmlfb2socHJlZFsiZXhwbGFuYXRpb25IaSJdKSBhbmQgZGV2YW5hZ2FyaV9vayhwcmVkWyJhY3Rpb25UaXBIaSJdKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgYW5kIGFsbCh3b3JkX2NvdW50KHByZWRba10pIDw9IEVYUExBTkFUSU9OX1dPUkRfQ0FQCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBrIGluICgiZXhwbGFuYXRpb25FbiIsICJleHBsYW5hdGlvbkhpIiwgImFjdGlvblRpcEVuIiwgImFjdGlvblRpcEhpIikpKQogICAgICAgICAgICBpZiBnb2xkWyJyaXNrTGV2ZWwiXSA9PSAiSElHSF9SSVNLIjoKICAgICAgICAgICAgICAgIHNjYW1fbiArPSAxCiAgICAgICAgICAgICAgICBzX29rID0gdmFsaWQgYW5kIHByZWRbInJpc2tMZXZlbCJdID09ICJISUdIX1JJU0siIGFuZCAiMTkzMCIgaW4gcHJlZFsiYWN0aW9uVGlwRW4iXQogICAgICAgICAgICAgICAgc2NhbV9oaXQgKz0gc19vawogICAgICAgICAgICAgICAgb2sgPSBzX29rCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBvayA9IHZhbGlkIGFuZCBwcmVkWyJyaXNrTGV2ZWwiXSA9PSBnb2xkWyJyaXNrTGV2ZWwiXQogICAgICAgICAgICAgICAgaWYgZ29sZFsicmlza0xldmVsIl0gPT0gIlNBRkUiOgogICAgICAgICAgICAgICAgICAgIHNhZmVfbiArPSAxCiAgICAgICAgICAgICAgICAgICAgaWYgdmFsaWQgYW5kIHByZWRbInJpc2tMZXZlbCJdICE9ICJTQUZFIjoKICAgICAgICAgICAgICAgICAgICAgICAgc2FmZV9mYWxzZV9hbGFybSArPSAxCgogICAgICAgIHBlcl9idWNrZXRbcm93WyJidWNrZXQiXV1bMF0gKz0gb2sKICAgICAgICBwZXJfYnVja2V0W3Jvd1siYnVja2V0Il1dWzFdICs9IDEKICAgICAgICBpZiBub3Qgb2sgYW5kIGxlbihmYWlsdXJlcykgPCAyNToKICAgICAgICAgICAgZmFpbHVyZXMuYXBwZW5kKHsiaWQiOiByb3cuZ2V0KCJpZCIpLCAiYnVja2V0Ijogcm93WyJidWNrZXQiXSwgInF1ZXJ5Ijogcm93WyJxdWVyeSJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICJnb2xkIjogcm93WyJjb21wbGV0aW9uIl0sICJvdXRwdXQiOiByYXdbOjQwMF19KQoKICAgIHJlcG9ydCA9IHsKICAgICAgICAibW9kZSI6ICJrcGkiLAogICAgICAgICJldmFsX3Jvd3MiOiBsZW4oZXZhbHMpLAogICAgICAgICJ2YWxpZF9qc29uX3BjdCI6IHBjdChuX3ZhbGlkLCBsZW4oZXZhbHMpKSwKICAgICAgICAiaW50ZW50X2FjY3VyYWN5X3BjdCI6IHBjdChpbnRlbnRfaGl0LCBpbnRlbnRfbiksCiAgICAgICAgInNjYW1ndWFyZF9yZWNhbGxfcGN0IjogcGN0KHNjYW1faGl0LCBzY2FtX24pLAogICAgICAgICJzYWZlX2Fic3RlbnRpb25fcGN0IjogcGN0KGFic3RfaGl0LCBhYnN0X24pLAogICAgICAgICJiaWxpbmd1YWxfdHRzX3BjdCI6IHBjdCh0dHNfaGl0LCB0dHNfbiksCiAgICAgICAgIm92ZXJfYWJzdGVudGlvbl9wY3QiOiBwY3Qob3Zlcl9hYnN0YWluLCBhbnN3ZXJhYmxlX24pLAogICAgICAgICJzY2FtX3NhZmVfZmFsc2VfYWxhcm1fcGN0IjogcGN0KHNhZmVfZmFsc2VfYWxhcm0sIHNhZmVfbiksCiAgICAgICAgInBlcl9idWNrZXRfYWNjdXJhY3lfcGN0Ijoge2I6IHBjdChoLCBuKSBmb3IgYiwgKGgsIG4pIGluIHNvcnRlZChwZXJfYnVja2V0Lml0ZW1zKCkpfSwKICAgICAgICAiZmFpbHVyZXNfc2FtcGxlIjogZmFpbHVyZXMsCiAgICB9CiAgICByZXBvcnRbImdhdGVzIl0gPSB7ZiJ7a30gPj0ge3Z9IjogcmVwb3J0W2tdID49IHYgZm9yIGssIHYgaW4gS1BJX0dBVEVTLml0ZW1zKCl9CiAgICByZXBvcnRbInBhc3NlZCJdID0gYWxsKHJlcG9ydFsiZ2F0ZXMiXS52YWx1ZXMoKSkKICAgIHJldHVybiByZXBvcnQKCgpkZWYgZ2VuZXJhdGVfb3V0cHV0cyhtb2RlbF9kaXI6IHN0ciwgZXZhbHM6IGxpc3RbZGljdF0sIGJhdGNoX3NpemU6IGludCwgbWF4X25ld190b2tlbnM6IGludCkgLT4gbGlzdFtzdHJdOgogICAgaW1wb3J0IHRvcmNoCiAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b01vZGVsRm9yQ2F1c2FsTE0sIEF1dG9Ub2tlbml6ZXIKCiAgICB0b2sgPSBBdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZChtb2RlbF9kaXIpCiAgICB0b2sucGFkZGluZ19zaWRlID0gImxlZnQiCiAgICBpZiB0b2sucGFkX3Rva2VuIGlzIE5vbmU6CiAgICAgICAgdG9rLnBhZF90b2tlbiA9IHRvay5lb3NfdG9rZW4KICAgIGR0eXBlID0gdG9yY2guYmZsb2F0MTYgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBhbmQgdG9yY2guY3VkYS5pc19iZjE2X3N1cHBvcnRlZCgpIGVsc2UgdG9yY2guZmxvYXQxNgogICAgaWYgbm90IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgZHR5cGUgPSB0b3JjaC5mbG9hdDMyCiAgICBpbXBvcnQgdHJhbnNmb3JtZXJzCgogICAgbWFqb3IsIG1pbm9yID0gKGludCh4KSBmb3IgeCBpbiB0cmFuc2Zvcm1lcnMuX192ZXJzaW9uX18uc3BsaXQoIi4iKVs6Ml0pCiAgICBkdHlwZV9rZXkgPSAiZHR5cGUiIGlmIChtYWpvciwgbWlub3IpID49ICg0LCA1NikgZWxzZSAidG9yY2hfZHR5cGUiICAjIHJlbmFtZWQgaW4gNC41NgogICAgbW9kZWwgPSBBdXRvTW9kZWxGb3JDYXVzYWxMTS5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgbW9kZWxfZGlyLCBkZXZpY2VfbWFwPSJhdXRvIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgTm9uZSwKICAgICAgICBhdHRuX2ltcGxlbWVudGF0aW9uPSJlYWdlciIsICoqe2R0eXBlX2tleTogZHR5cGV9LAogICAgKQogICAgbW9kZWwuZXZhbCgpCiAgICBzdG9wX2lkcyA9IFt0b2suZW9zX3Rva2VuX2lkLCB0b2suY29udmVydF90b2tlbnNfdG9faWRzKEVORF9UVVJOKV0KCiAgICBvdXRwdXRzID0gW10KICAgIGZvciBpIGluIHJhbmdlKDAsIGxlbihldmFscyksIGJhdGNoX3NpemUpOgogICAgICAgIGJhdGNoID0gZXZhbHNbaTppICsgYmF0Y2hfc2l6ZV0KICAgICAgICBwcm9tcHRzID0gW3dyYXBfcHJvbXB0KHJbInByb21wdCJdKSBmb3IgciBpbiBiYXRjaF0KICAgICAgICBlbmMgPSB0b2socHJvbXB0cywgcmV0dXJuX3RlbnNvcnM9InB0IiwgcGFkZGluZz1UcnVlLCBhZGRfc3BlY2lhbF90b2tlbnM9VHJ1ZSkudG8obW9kZWwuZGV2aWNlKQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBnZW4gPSBtb2RlbC5nZW5lcmF0ZSgqKmVuYywgbWF4X25ld190b2tlbnM9bWF4X25ld190b2tlbnMsIGRvX3NhbXBsZT1GYWxzZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZW9zX3Rva2VuX2lkPXN0b3BfaWRzLCBwYWRfdG9rZW5faWQ9dG9rLnBhZF90b2tlbl9pZCkKICAgICAgICBmb3Igc2VxIGluIGdlbls6LCBlbmNbImlucHV0X2lkcyJdLnNoYXBlWzFdOl06CiAgICAgICAgICAgIG91dHB1dHMuYXBwZW5kKHRvay5kZWNvZGUoc2VxLCBza2lwX3NwZWNpYWxfdG9rZW5zPVRydWUpLnN0cmlwKCkpCiAgICAgICAgcHJpbnQoZiIgIGdlbmVyYXRlZCB7bWluKGkgKyBiYXRjaF9zaXplLCBsZW4oZXZhbHMpKX0ve2xlbihldmFscyl9IiwgZmx1c2g9VHJ1ZSkKICAgIHJldHVybiBvdXRwdXRzCgoKZGVmIHByaW50X3JlcG9ydChyZXBvcnQ6IGRpY3QpIC0+IE5vbmU6CiAgICBoaWRkZW4gPSB7InByb2JsZW1zX3NhbXBsZSIsICJmYWlsdXJlc19zYW1wbGUifQogICAgcHJpbnQoanNvbi5kdW1wcyh7azogdiBmb3IgaywgdiBpbiByZXBvcnQuaXRlbXMoKSBpZiBrIG5vdCBpbiBoaWRkZW59LCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0yKSkKICAgIHNhbXBsZSA9IHJlcG9ydC5nZXQoInByb2JsZW1zX3NhbXBsZSIpIG9yIHJlcG9ydC5nZXQoImZhaWx1cmVzX3NhbXBsZSIpCiAgICBpZiBzYW1wbGU6CiAgICAgICAgcHJpbnQoZiJcbkZpcnN0IHttaW4oNSwgbGVuKHNhbXBsZSkpfSBwcm9ibGVtczoiKQogICAgICAgIGZvciBwIGluIHNhbXBsZVs6NV06CiAgICAgICAgICAgIHByaW50KCIgICIsIGpzb24uZHVtcHMocCwgZW5zdXJlX2FzY2lpPUZhbHNlKVs6NTAwXSkKICAgIHByaW50KCJcblJFU1VMVDoiLCAiUEFTUyIgaWYgcmVwb3J0WyJwYXNzZWQiXSBlbHNlICJGQUlMIikKCgpkZWYgbWFpbihhcmd2PU5vbmUpIC0+IGludDoKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXywgZm9ybWF0dGVyX2NsYXNzPWFyZ3BhcnNlLlJhd0Rlc2NyaXB0aW9uSGVscEZvcm1hdHRlcikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS10cmFpbi1maWxlIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1ldmFsLWZpbGUiLCByZXF1aXJlZD1UcnVlKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1vZGVsLWRpciIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcHJlZGljdGlvbnMtZmlsZSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2F2ZS1wcmVkaWN0aW9ucyIsIGhlbHA9IndyaXRlIHJhdyBtb2RlbCBvdXRwdXRzIGhlcmUgKG1vZGVsIG1vZGUpIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1yZXBvcnQtZmlsZSIsIGhlbHA9IndyaXRlIHRoZSBKU09OIHJlcG9ydCBoZXJlIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1saW1pdCIsIHR5cGU9aW50LCBoZWxwPSJzY29yZSBvbmx5IHRoZSBmaXJzdCBOIGV2YWwgcm93cyIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYmF0Y2gtc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTgpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbWF4LW5ldy10b2tlbnMiLCB0eXBlPWludCwgZGVmYXVsdD0yNTYpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncyhhcmd2KQoKICAgIGV2YWxzID0gcmVhZF9qc29ubChhcmdzLmV2YWxfZmlsZSkKICAgIGlmIGFyZ3MubGltaXQ6CiAgICAgICAgZXZhbHMgPSBldmFsc1s6IGFyZ3MubGltaXRdCgogICAgaWYgYXJncy5tb2RlbF9kaXIgb3IgYXJncy5wcmVkaWN0aW9uc19maWxlOgogICAgICAgIGlmIGFyZ3MucHJlZGljdGlvbnNfZmlsZToKICAgICAgICAgICAgcHJlZHMgPSByZWFkX2pzb25sKGFyZ3MucHJlZGljdGlvbnNfZmlsZSkKICAgICAgICAgICAgYnlfaWQgPSB7cFsiaWQiXTogcFsib3V0cHV0Il0gZm9yIHAgaW4gcHJlZHMgaWYgImlkIiBpbiBwfQogICAgICAgICAgICBvdXRwdXRzID0gW2J5X2lkW3JbImlkIl1dIGZvciByIGluIGV2YWxzXSBpZiBieV9pZCBlbHNlIFtwWyJvdXRwdXQiXSBmb3IgcCBpbiBwcmVkc1s6IGxlbihldmFscyldXQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIG91dHB1dHMgPSBnZW5lcmF0ZV9vdXRwdXRzKGFyZ3MubW9kZWxfZGlyLCBldmFscywgYXJncy5iYXRjaF9zaXplLCBhcmdzLm1heF9uZXdfdG9rZW5zKQogICAgICAgICAgICBpZiBhcmdzLnNhdmVfcHJlZGljdGlvbnM6CiAgICAgICAgICAgICAgICB3aXRoIG9wZW4oYXJncy5zYXZlX3ByZWRpY3Rpb25zLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgICAgICAgICAgICAgZm9yIHIsIG8gaW4gemlwKGV2YWxzLCBvdXRwdXRzKToKICAgICAgICAgICAgICAgICAgICAgICAgZi53cml0ZShqc29uLmR1bXBzKHsiaWQiOiByWyJpZCJdLCAib3V0cHV0Ijogb30sIGVuc3VyZV9hc2NpaT1GYWxzZSkgKyAiXG4iKQogICAgICAgIHJlcG9ydCA9IHNjb3JlX291dHB1dHMoZXZhbHMsIG91dHB1dHMpCiAgICBlbHNlOgogICAgICAgIGlmIG5vdCBhcmdzLnRyYWluX2ZpbGU6CiAgICAgICAgICAgIGFwLmVycm9yKCJkYXRhc2V0IGF1ZGl0IG5lZWRzIC0tdHJhaW4tZmlsZSAob3IgcGFzcyAtLW1vZGVsLWRpciAvIC0tcHJlZGljdGlvbnMtZmlsZSkiKQogICAgICAgIHJlcG9ydCA9IGF1ZGl0X2RhdGFzZXQocmVhZF9qc29ubChhcmdzLnRyYWluX2ZpbGUpLCBldmFscykKCiAgICBwcmludF9yZXBvcnQocmVwb3J0KQogICAgaWYgYXJncy5yZXBvcnRfZmlsZToKICAgICAgICB3aXRoIG9wZW4oYXJncy5yZXBvcnRfZmlsZSwgInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgICAgICBqc29uLmR1bXAocmVwb3J0LCBmLCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0yKQogICAgcmV0dXJuIDAgaWYgcmVwb3J0WyJwYXNzZWQiXSBlbHNlIDEKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgc3lzLmV4aXQobWFpbigpKQo=',
    'train_gemma2b_qlora.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJRTG9SQSBmaW5lLXR1bmUgb2YgZ29vZ2xlL2dlbW1hLTItMmItaXQgb24gYSBmcmVlIENvbGFiIFQ0IChndWlkZSBwYWdlIDYpLgoKICAqIDQtYml0IE5GNCArIGRvdWJsZSBxdWFudCBiYXNlLCBhdHRuX2ltcGxlbWVudGF0aW9uPSJlYWdlciIgKEdlbW1hLTIgc29mdGNhcHBpbmcpCiAgKiBMb1JBIHI9MTYsIGFscGhhPTMyLCBkcm9wb3V0PTAuMDUgb24gcS9rL3Yvby9nYXRlL3VwL2Rvd24gcHJvamVjdGlvbnMKICAqIGJzPTIgeCBncmFkX2FjY3VtPTggKGVmZmVjdGl2ZSAxNiksIHBhZ2VkX2FkYW13XzhiaXQsIGdyYWRpZW50IGNoZWNrcG9pbnRpbmcKICAqIGxyPTJlLTQgY29zaW5lLCB3YXJtdXAgOCUsIDQgZXBvY2hzLCBtYXhfc2VxX2xlbj01MTIgKD0gTGxtSW5mZXJlbmNlLnNldE1heFRva2Vucyg1MTIpKQogICogY29tcGxldGlvbi1vbmx5IGxvc3M6IGV2ZXJ5IHRva2VuIHVwIHRvIGFuZCBpbmNsdWRpbmcgIjxzdGFydF9vZl90dXJuPm1vZGVsXFxuIiBnZXRzIGxhYmVsIC0xMDAsCiAgICBzbyBsb3NzIGlzIGNvbXB1dGVkIG9uIHRoZSBKU09OIGFuc3dlciArIDxlbmRfb2ZfdHVybj4gb25seSAoc2FtZSBlZmZlY3QgYXMgVFJMJ3MKICAgIERhdGFDb2xsYXRvckZvckNvbXBsZXRpb25Pbmx5TE0sIGltcGxlbWVudGVkIGRpcmVjdGx5IHNvIGl0IHdvcmtzIGFjcm9zcyBUUkwgdmVyc2lvbnMpCiAgKiBtZXJnZV9hbmRfdW5sb2FkKCkgLT4gPG91dHB1dC1kaXI+L21lcmdlZF9mcDE2IChtb2RlbC5zYWZldGVuc29ycyArIHRva2VuaXplci5tb2RlbCkKCiAgcHl0aG9uMyB0cmFpbl9nZW1tYTJiX3Fsb3JhLnB5IC0tbW9kZWwtaWQgZ29vZ2xlL2dlbW1hLTItMmItaXQgXFwKICAgICAgLS10cmFpbi1maWxlIGRhdGEvdHJhaW4uanNvbmwgLS1ldmFsLWZpbGUgZGF0YS9ldmFsLmpzb25sIFxcCiAgICAgIC0tZXBvY2hzIDQgLS1vdXRwdXQtZGlyIG91dHB1dHMvZ2VtbWEyLTJiLWJhbmtzYXRoaQoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBnYwppbXBvcnQgaW5zcGVjdAppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHNodXRpbAppbXBvcnQgc3lzCgpmcm9tIGJhbmtzYXRoaV9jb21tb24gaW1wb3J0IEVORF9UVVJOLCBNT0RFTF9UVVJOLCByZWFkX2pzb25sCgpMT1JBX1RBUkdFVFMgPSBbInFfcHJvaiIsICJrX3Byb2oiLCAidl9wcm9qIiwgIm9fcHJvaiIsICJnYXRlX3Byb2oiLCAidXBfcHJvaiIsICJkb3duX3Byb2oiXQoKCmRlZiBkdHlwZV9rd2FyZyhkdHlwZSkgLT4gZGljdDoKICAgICIiInRyYW5zZm9ybWVycyA+PSA0LjU2IHJlbmFtZWQgZnJvbV9wcmV0cmFpbmVkKHRvcmNoX2R0eXBlPS4uLikgdG8gZHR5cGU9Li4uIiIiCiAgICBpbXBvcnQgdHJhbnNmb3JtZXJzCgogICAgbWFqb3IsIG1pbm9yID0gKGludCh4KSBmb3IgeCBpbiB0cmFuc2Zvcm1lcnMuX192ZXJzaW9uX18uc3BsaXQoIi4iKVs6Ml0pCiAgICByZXR1cm4geyJkdHlwZSI6IGR0eXBlfSBpZiAobWFqb3IsIG1pbm9yKSA+PSAoNCwgNTYpIGVsc2UgeyJ0b3JjaF9kdHlwZSI6IGR0eXBlfQoKCmRlZiBwYXJzZV9hcmdzKGFyZ3Y9Tm9uZSk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18sIGZvcm1hdHRlcl9jbGFzcz1hcmdwYXJzZS5SYXdEZXNjcmlwdGlvbkhlbHBGb3JtYXR0ZXIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbW9kZWwtaWQiLCBkZWZhdWx0PSJnb29nbGUvZ2VtbWEtMi0yYi1pdCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdHJhaW4tZmlsZSIsIGRlZmF1bHQ9ImRhdGEvdHJhaW4uanNvbmwiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWV2YWwtZmlsZSIsIGRlZmF1bHQ9ImRhdGEvZXZhbC5qc29ubCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0cHV0LWRpciIsIGRlZmF1bHQ9Im91dHB1dHMvZ2VtbWEyLTJiLWJhbmtzYXRoaSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZXBvY2hzIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD00KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWxyIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0yZS00KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWJhdGNoLXNpemUiLCB0eXBlPWludCwgZGVmYXVsdD0yKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWdyYWQtYWNjdW0iLCB0eXBlPWludCwgZGVmYXVsdD04KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXdhcm11cC1yYXRpbyIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4wOCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1tYXgtc2VxLWxlbiIsIHR5cGU9aW50LCBkZWZhdWx0PTUxMikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1sb3JhLXIiLCB0eXBlPWludCwgZGVmYXVsdD0xNikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1sb3JhLWFscGhhIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MzIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbG9yYS1kcm9wb3V0IiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjA1KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNlZWQiLCB0eXBlPWludCwgZGVmYXVsdD00MikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1uby1tZXJnZSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsIGhlbHA9InNraXAgbWVyZ2VfYW5kX3VubG9hZCAoYWRhcHRlciBvbmx5KSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbWVyZ2Utb25seSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsIGhlbHA9Im9ubHkgbWVyZ2UgYW4gZXhpc3RpbmcgYWRhcHRlciBpbiBvdXRwdXQtZGlyL2FkYXB0ZXIiKQogICAgcmV0dXJuIGFwLnBhcnNlX2FyZ3MoYXJndikKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIFRva2VuaXNhdGlvbiB3aXRoIGNvbXBsZXRpb24tb25seSBsYWJlbCBtYXNraW5nCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCmRlZiBidWlsZF9mZWF0dXJlcyhyb3dzLCB0b2ssIG1heF9sZW4pOgogICAgZmVhdHMsIGRyb3BwZWQgPSBbXSwgMAogICAgZm9yIHIgaW4gcm93czoKICAgICAgICB0ZXh0ID0gclsidGV4dCJdCiAgICAgICAgY3V0ID0gdGV4dC5pbmRleChNT0RFTF9UVVJOKSArIGxlbihNT0RFTF9UVVJOKSAgIyB0ZW1wbGF0ZSBib3VuZGFyeQogICAgICAgIHByb21wdF9pZHMgPSB0b2sodGV4dFs6Y3V0XSwgYWRkX3NwZWNpYWxfdG9rZW5zPVRydWUpWyJpbnB1dF9pZHMiXSAgIyBhZGRzIDxib3M+CiAgICAgICAgY29tcGxldGlvbl9pZHMgPSB0b2sodGV4dFtjdXQ6XSwgYWRkX3NwZWNpYWxfdG9rZW5zPUZhbHNlKVsiaW5wdXRfaWRzIl0KICAgICAgICBpZHMgPSBwcm9tcHRfaWRzICsgY29tcGxldGlvbl9pZHMKICAgICAgICBpZiBsZW4oaWRzKSA+IG1heF9sZW46ICAjIG5ldmVyIHRydW5jYXRlIHRoZSBKU09OIHRhcmdldCAtIGRyb3AgaW5zdGVhZAogICAgICAgICAgICBkcm9wcGVkICs9IDEKICAgICAgICAgICAgY29udGludWUKICAgICAgICBmZWF0cy5hcHBlbmQoewogICAgICAgICAgICAiaW5wdXRfaWRzIjogaWRzLAogICAgICAgICAgICAiYXR0ZW50aW9uX21hc2siOiBbMV0gKiBsZW4oaWRzKSwKICAgICAgICAgICAgImxhYmVscyI6IFstMTAwXSAqIGxlbihwcm9tcHRfaWRzKSArIGNvbXBsZXRpb25faWRzLAogICAgICAgIH0pCiAgICByZXR1cm4gZmVhdHMsIGRyb3BwZWQKCgpjbGFzcyBDb21wbGV0aW9uT25seUNvbGxhdG9yOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHBhZF9pZDogaW50KToKICAgICAgICBzZWxmLnBhZF9pZCA9IHBhZF9pZAoKICAgIGRlZiBfX2NhbGxfXyhzZWxmLCBiYXRjaCk6CiAgICAgICAgaW1wb3J0IHRvcmNoCgogICAgICAgIHdpZHRoID0gbWF4KGxlbihiWyJpbnB1dF9pZHMiXSkgZm9yIGIgaW4gYmF0Y2gpCiAgICAgICAgb3V0ID0geyJpbnB1dF9pZHMiOiBbXSwgImF0dGVudGlvbl9tYXNrIjogW10sICJsYWJlbHMiOiBbXX0KICAgICAgICBmb3IgYiBpbiBiYXRjaDoKICAgICAgICAgICAgcGFkID0gd2lkdGggLSBsZW4oYlsiaW5wdXRfaWRzIl0pCiAgICAgICAgICAgIG91dFsiaW5wdXRfaWRzIl0uYXBwZW5kKGJbImlucHV0X2lkcyJdICsgW3NlbGYucGFkX2lkXSAqIHBhZCkKICAgICAgICAgICAgb3V0WyJhdHRlbnRpb25fbWFzayJdLmFwcGVuZChiWyJhdHRlbnRpb25fbWFzayJdICsgWzBdICogcGFkKQogICAgICAgICAgICBvdXRbImxhYmVscyJdLmFwcGVuZChiWyJsYWJlbHMiXSArIFstMTAwXSAqIHBhZCkKICAgICAgICByZXR1cm4ge2s6IHRvcmNoLnRlbnNvcih2LCBkdHlwZT10b3JjaC5sb25nKSBmb3IgaywgdiBpbiBvdXQuaXRlbXMoKX0KCgpkZWYgc2FuaXR5X2NoZWNrX21hc2tpbmcoZmVhdHMsIHRvayk6CiAgICBmID0gZmVhdHNbMF0KICAgIHRyYWluZWQgPSBbdCBmb3IgdCwgbCBpbiB6aXAoZlsiaW5wdXRfaWRzIl0sIGZbImxhYmVscyJdKSBpZiBsICE9IC0xMDBdCiAgICBkZWNvZGVkID0gdG9rLmRlY29kZSh0cmFpbmVkKQogICAgYXNzZXJ0IGRlY29kZWQubHN0cmlwKCkuc3RhcnRzd2l0aCgieyIpLCBmImxvc3Mgc3BhbiBzaG91bGQgc3RhcnQgYXQgSlNPTiwgZ290OiB7ZGVjb2RlZFs6ODBdIXJ9IgogICAgYXNzZXJ0IEVORF9UVVJOIGluIGRlY29kZWQsICJsb3NzIHNwYW4gbXVzdCBpbmNsdWRlIDxlbmRfb2ZfdHVybj4gc28gdGhlIG1vZGVsIGxlYXJucyB0byBzdG9wIgogICAgcHJpbnQoZiJbbWFzayBjaGVja10gdHJhaW5lZCBzcGFuOiB7ZGVjb2RlZFs6MTIwXX0uLi4iKQoKCmRlZiB0cmFpbmluZ19hcmdzKGFyZ3MsIGZwMTY6IGJvb2wsIGJmMTY6IGJvb2wsIG5fdHJhaW46IGludCk6CiAgICBpbXBvcnQgbWF0aAoKICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBUcmFpbmluZ0FyZ3VtZW50cwoKICAgIHN0ZXBzX3Blcl9lcG9jaCA9IG1hdGguY2VpbChuX3RyYWluIC8gKGFyZ3MuYmF0Y2hfc2l6ZSAqIGFyZ3MuZ3JhZF9hY2N1bSkpCiAgICB0b3RhbF9zdGVwcyA9IG1heCgxLCBtYXRoLmNlaWwoc3RlcHNfcGVyX2Vwb2NoICogYXJncy5lcG9jaHMpKQogICAga3dhcmdzID0gZGljdCgKICAgICAgICBvdXRwdXRfZGlyPW9zLnBhdGguam9pbihhcmdzLm91dHB1dF9kaXIsICJjaGVja3BvaW50cyIpLAogICAgICAgIHBlcl9kZXZpY2VfdHJhaW5fYmF0Y2hfc2l6ZT1hcmdzLmJhdGNoX3NpemUsCiAgICAgICAgcGVyX2RldmljZV9ldmFsX2JhdGNoX3NpemU9YXJncy5iYXRjaF9zaXplLAogICAgICAgIGdyYWRpZW50X2FjY3VtdWxhdGlvbl9zdGVwcz1hcmdzLmdyYWRfYWNjdW0sCiAgICAgICAgbnVtX3RyYWluX2Vwb2Nocz1hcmdzLmVwb2NocywKICAgICAgICBsZWFybmluZ19yYXRlPWFyZ3MubHIsCiAgICAgICAgbHJfc2NoZWR1bGVyX3R5cGU9ImNvc2luZSIsCiAgICAgICAgd2FybXVwX3N0ZXBzPW1heCgxLCByb3VuZCh0b3RhbF9zdGVwcyAqIGFyZ3Mud2FybXVwX3JhdGlvKSksICAjIDglIHdhcm11cCAod2FybXVwX3JhdGlvIHJlbW92ZWQgaW4gdHJhbnNmb3JtZXJzIDUpCiAgICAgICAgb3B0aW09InBhZ2VkX2FkYW13XzhiaXQiLAogICAgICAgIHdlaWdodF9kZWNheT0wLjAsCiAgICAgICAgbWF4X2dyYWRfbm9ybT0wLjMsCiAgICAgICAgZnAxNj1mcDE2LAogICAgICAgIGJmMTY9YmYxNiwKICAgICAgICBsb2dnaW5nX3N0ZXBzPTEwLAogICAgICAgIHNhdmVfc3RyYXRlZ3k9ImVwb2NoIiwKICAgICAgICBzYXZlX3RvdGFsX2xpbWl0PTIsCiAgICAgICAgbG9hZF9iZXN0X21vZGVsX2F0X2VuZD1UcnVlLAogICAgICAgIG1ldHJpY19mb3JfYmVzdF9tb2RlbD0iZXZhbF9sb3NzIiwKICAgICAgICBncmVhdGVyX2lzX2JldHRlcj1GYWxzZSwKICAgICAgICBncmFkaWVudF9jaGVja3BvaW50aW5nPVRydWUsCiAgICAgICAgZ3JhZGllbnRfY2hlY2twb2ludGluZ19rd2FyZ3M9eyJ1c2VfcmVlbnRyYW50IjogRmFsc2V9LAogICAgICAgIHJlcG9ydF90bz0ibm9uZSIsCiAgICAgICAgc2VlZD1hcmdzLnNlZWQsCiAgICAgICAgcmVtb3ZlX3VudXNlZF9jb2x1bW5zPUZhbHNlLAogICAgKQogICAgIyB0cmFuc2Zvcm1lcnMgcmVuYW1lZCBldmFsdWF0aW9uX3N0cmF0ZWd5IC0+IGV2YWxfc3RyYXRlZ3kgaW4gNC40MQogICAgcGFyYW1zID0gaW5zcGVjdC5zaWduYXR1cmUoVHJhaW5pbmdBcmd1bWVudHMuX19pbml0X18pLnBhcmFtZXRlcnMKICAgIGt3YXJnc1siZXZhbF9zdHJhdGVneSIgaWYgImV2YWxfc3RyYXRlZ3kiIGluIHBhcmFtcyBlbHNlICJldmFsdWF0aW9uX3N0cmF0ZWd5Il0gPSAiZXBvY2giCiAgICBkcm9wcGVkID0gc29ydGVkKGsgZm9yIGsgaW4ga3dhcmdzIGlmIGsgbm90IGluIHBhcmFtcykKICAgIGlmIGRyb3BwZWQ6CiAgICAgICAgcHJpbnQoZiJub3RlOiB0aGlzIHRyYW5zZm9ybWVycyB2ZXJzaW9uIGRvZXMgbm90IGFjY2VwdCB7ZHJvcHBlZH07IHNraXBwaW5nIHRoZW0iKQogICAgcHJpbnQoZiJzY2hlZHVsZToge3RvdGFsX3N0ZXBzfSBvcHRpbWl6ZXIgc3RlcHMsIHdhcm11cCB7a3dhcmdzWyd3YXJtdXBfc3RlcHMnXX0iKQogICAgcmV0dXJuIFRyYWluaW5nQXJndW1lbnRzKCoqe2s6IHYgZm9yIGssIHYgaW4ga3dhcmdzLml0ZW1zKCkgaWYgayBpbiBwYXJhbXN9KQoKCmRlZiB0cmFpbihhcmdzKToKICAgIGltcG9ydCB0b3JjaAogICAgZnJvbSBkYXRhc2V0cyBpbXBvcnQgRGF0YXNldAogICAgZnJvbSBwZWZ0IGltcG9ydCBMb3JhQ29uZmlnLCBnZXRfcGVmdF9tb2RlbCwgcHJlcGFyZV9tb2RlbF9mb3Jfa2JpdF90cmFpbmluZwogICAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEF1dG9Nb2RlbEZvckNhdXNhbExNLCBBdXRvVG9rZW5pemVyLCBCaXRzQW5kQnl0ZXNDb25maWcsIFRyYWluZXIsIHNldF9zZWVkCgogICAgc2V0X3NlZWQoYXJncy5zZWVkKQogICAgaWYgbm90IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgc3lzLmV4aXQoIkEgQ1VEQSBHUFUgaXMgcmVxdWlyZWQgKENvbGFiOiBSdW50aW1lIC0+IENoYW5nZSBydW50aW1lIHR5cGUgLT4gVDQgR1BVKS4iKQogICAgYmYxNiA9IHRvcmNoLmN1ZGEuaXNfYmYxNl9zdXBwb3J0ZWQoKSBhbmQgdG9yY2guY3VkYS5nZXRfZGV2aWNlX2NhcGFiaWxpdHkoKVswXSA+PSA4CiAgICBjb21wdXRlX2R0eXBlID0gdG9yY2guYmZsb2F0MTYgaWYgYmYxNiBlbHNlIHRvcmNoLmZsb2F0MTYgICMgVDQgKHNtXzc1KSAtPiBmcDE2CiAgICBwcmludChmIkdQVToge3RvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApfSB8IGNvbXB1dGUgZHR5cGU6IHtjb21wdXRlX2R0eXBlfSIpCgogICAgdG9rID0gQXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQoYXJncy5tb2RlbF9pZCkKICAgIHRvay5wYWRkaW5nX3NpZGUgPSAicmlnaHQiCiAgICBpZiB0b2sucGFkX3Rva2VuIGlzIE5vbmU6CiAgICAgICAgdG9rLnBhZF90b2tlbiA9IHRvay5lb3NfdG9rZW4KCiAgICB0cmFpbl9yb3dzLCBldmFsX3Jvd3MgPSByZWFkX2pzb25sKGFyZ3MudHJhaW5fZmlsZSksIHJlYWRfanNvbmwoYXJncy5ldmFsX2ZpbGUpCiAgICB0cmFpbl9mZWF0cywgZDEgPSBidWlsZF9mZWF0dXJlcyh0cmFpbl9yb3dzLCB0b2ssIGFyZ3MubWF4X3NlcV9sZW4pCiAgICBldmFsX2ZlYXRzLCBkMiA9IGJ1aWxkX2ZlYXR1cmVzKGV2YWxfcm93cywgdG9rLCBhcmdzLm1heF9zZXFfbGVuKQogICAgbGVuZ3RocyA9IFtsZW4oZlsiaW5wdXRfaWRzIl0pIGZvciBmIGluIHRyYWluX2ZlYXRzXQogICAgcHJpbnQoZiJ0cmFpbiB7bGVuKHRyYWluX2ZlYXRzKX0gKGRyb3BwZWQge2QxfSA+IHthcmdzLm1heF9zZXFfbGVufSB0b2spIHwgZXZhbCB7bGVuKGV2YWxfZmVhdHMpfSAoZHJvcHBlZCB7ZDJ9KSIKICAgICAgICAgIGYiIHwgbWF4IGxlbiB7bWF4KGxlbmd0aHMpfSB8IG1lYW4gbGVuIHtzdW0obGVuZ3RocykgLyBsZW4obGVuZ3Rocyk6LjBmfSIpCiAgICBzYW5pdHlfY2hlY2tfbWFza2luZyh0cmFpbl9mZWF0cywgdG9rKQoKICAgIGJuYiA9IEJpdHNBbmRCeXRlc0NvbmZpZygKICAgICAgICBsb2FkX2luXzRiaXQ9VHJ1ZSwKICAgICAgICBibmJfNGJpdF9xdWFudF90eXBlPSJuZjQiLAogICAgICAgIGJuYl80Yml0X3VzZV9kb3VibGVfcXVhbnQ9VHJ1ZSwKICAgICAgICBibmJfNGJpdF9jb21wdXRlX2R0eXBlPWNvbXB1dGVfZHR5cGUsCiAgICApCiAgICBtb2RlbCA9IEF1dG9Nb2RlbEZvckNhdXNhbExNLmZyb21fcHJldHJhaW5lZCgKICAgICAgICBhcmdzLm1vZGVsX2lkLAogICAgICAgIHF1YW50aXphdGlvbl9jb25maWc9Ym5iLAogICAgICAgIGRldmljZV9tYXA9eyIiOiAwfSwKICAgICAgICBhdHRuX2ltcGxlbWVudGF0aW9uPSJlYWdlciIsICAjIG1hbmRhdG9yeSBmb3IgR2VtbWEtMiBsb2dpdCBzb2Z0Y2FwcGluZyBkdXJpbmcgdHJhaW5pbmcKICAgICAgICAqKmR0eXBlX2t3YXJnKGNvbXB1dGVfZHR5cGUpLAogICAgKQogICAgbW9kZWwuY29uZmlnLnVzZV9jYWNoZSA9IEZhbHNlCiAgICBtb2RlbCA9IHByZXBhcmVfbW9kZWxfZm9yX2tiaXRfdHJhaW5pbmcobW9kZWwsIHVzZV9ncmFkaWVudF9jaGVja3BvaW50aW5nPVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZ3JhZGllbnRfY2hlY2twb2ludGluZ19rd2FyZ3M9eyJ1c2VfcmVlbnRyYW50IjogRmFsc2V9KQogICAgbG9yYSA9IExvcmFDb25maWcocj1hcmdzLmxvcmFfciwgbG9yYV9hbHBoYT1hcmdzLmxvcmFfYWxwaGEsIGxvcmFfZHJvcG91dD1hcmdzLmxvcmFfZHJvcG91dCwKICAgICAgICAgICAgICAgICAgICAgIHRhcmdldF9tb2R1bGVzPUxPUkFfVEFSR0VUUywgYmlhcz0ibm9uZSIsIHRhc2tfdHlwZT0iQ0FVU0FMX0xNIikKICAgIG1vZGVsID0gZ2V0X3BlZnRfbW9kZWwobW9kZWwsIGxvcmEpCiAgICBtb2RlbC5wcmludF90cmFpbmFibGVfcGFyYW1ldGVycygpCgogICAgdHJhaW5lciA9IFRyYWluZXIoCiAgICAgICAgbW9kZWw9bW9kZWwsCiAgICAgICAgYXJncz10cmFpbmluZ19hcmdzKGFyZ3MsIGZwMTY9bm90IGJmMTYsIGJmMTY9YmYxNiwgbl90cmFpbj1sZW4odHJhaW5fZmVhdHMpKSwKICAgICAgICB0cmFpbl9kYXRhc2V0PURhdGFzZXQuZnJvbV9saXN0KHRyYWluX2ZlYXRzKSwKICAgICAgICBldmFsX2RhdGFzZXQ9RGF0YXNldC5mcm9tX2xpc3QoZXZhbF9mZWF0cyksCiAgICAgICAgZGF0YV9jb2xsYXRvcj1Db21wbGV0aW9uT25seUNvbGxhdG9yKHRvay5wYWRfdG9rZW5faWQpLAogICAgKQogICAgcmVzdWx0ID0gdHJhaW5lci50cmFpbigpCiAgICBtZXRyaWNzID0gdHJhaW5lci5ldmFsdWF0ZSgpCiAgICBwZWFrX2diID0gdG9yY2guY3VkYS5tYXhfbWVtb3J5X2FsbG9jYXRlZCgpIC8gMTAyNCAqKiAzCiAgICBwcmludChmInBlYWsgVlJBTSBhbGxvY2F0ZWQ6IHtwZWFrX2diOi4yZn0gR0IgfCBldmFsX2xvc3M6IHttZXRyaWNzLmdldCgnZXZhbF9sb3NzJyl9IikKCiAgICBhZGFwdGVyX2RpciA9IG9zLnBhdGguam9pbihhcmdzLm91dHB1dF9kaXIsICJhZGFwdGVyIikKICAgIHRyYWluZXIubW9kZWwuc2F2ZV9wcmV0cmFpbmVkKGFkYXB0ZXJfZGlyKQogICAgdG9rLnNhdmVfcHJldHJhaW5lZChhZGFwdGVyX2RpcikKICAgIHdpdGggb3Blbihvcy5wYXRoLmpvaW4oYXJncy5vdXRwdXRfZGlyLCAidHJhaW5fbWV0cmljcy5qc29uIiksICJ3IikgYXMgZjoKICAgICAgICBqc29uLmR1bXAoeyJ0cmFpbiI6IHJlc3VsdC5tZXRyaWNzLCAiZXZhbCI6IG1ldHJpY3MsICJwZWFrX3ZyYW1fZ2IiOiByb3VuZChwZWFrX2diLCAyKSwKICAgICAgICAgICAgICAgICAgICJ0cmFpbl9yb3dzIjogbGVuKHRyYWluX2ZlYXRzKSwgImV2YWxfcm93cyI6IGxlbihldmFsX2ZlYXRzKSwKICAgICAgICAgICAgICAgICAgICJkcm9wcGVkX292ZXJsZW5ndGgiOiBkMSArIGQyLCAiYXJncyI6IHZhcnMoYXJncyl9LCBmLCBpbmRlbnQ9MikKICAgIHByaW50KGYiYWRhcHRlciBzYXZlZCAtPiB7YWRhcHRlcl9kaXJ9IikKCiAgICBkZWwgdHJhaW5lciwgbW9kZWwKICAgIGdjLmNvbGxlY3QoKQogICAgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpCgoKZGVmIG1lcmdlKGFyZ3MpOgogICAgaW1wb3J0IHRvcmNoCiAgICBmcm9tIHBlZnQgaW1wb3J0IFBlZnRNb2RlbAogICAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEF1dG9Nb2RlbEZvckNhdXNhbExNLCBBdXRvVG9rZW5pemVyCgogICAgYWRhcHRlcl9kaXIgPSBvcy5wYXRoLmpvaW4oYXJncy5vdXRwdXRfZGlyLCAiYWRhcHRlciIpCiAgICBtZXJnZWRfZGlyID0gb3MucGF0aC5qb2luKGFyZ3Mub3V0cHV0X2RpciwgIm1lcmdlZF9mcDE2IikKICAgIHByaW50KGYibWVyZ2luZyB7YWRhcHRlcl9kaXJ9IGludG8gZnAxNiBiYXNlIC4uLiIpCiAgICBiYXNlID0gQXV0b01vZGVsRm9yQ2F1c2FsTE0uZnJvbV9wcmV0cmFpbmVkKAogICAgICAgIGFyZ3MubW9kZWxfaWQsIGF0dG5faW1wbGVtZW50YXRpb249ImVhZ2VyIiwKICAgICAgICBkZXZpY2VfbWFwPSJhdXRvIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgTm9uZSwgKipkdHlwZV9rd2FyZyh0b3JjaC5mbG9hdDE2KSwKICAgICkKICAgIG1lcmdlZCA9IFBlZnRNb2RlbC5mcm9tX3ByZXRyYWluZWQoYmFzZSwgYWRhcHRlcl9kaXIpLm1lcmdlX2FuZF91bmxvYWQoKQogICAgbWVyZ2VkLmdlbmVyYXRpb25fY29uZmlnLmRvX3NhbXBsZSA9IEZhbHNlCiAgICBtZXJnZWQuc2F2ZV9wcmV0cmFpbmVkKG1lcmdlZF9kaXIsIG1heF9zaGFyZF9zaXplPSI2R0IiKSAgIyBzYWZldGVuc29ycyBieSBkZWZhdWx0CiAgICB0b2sgPSBBdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZChhcmdzLm1vZGVsX2lkKQogICAgdG9rLnNhdmVfcHJldHJhaW5lZChtZXJnZWRfZGlyKQoKICAgICMgTWVkaWFQaXBlIGJ1bmRsZXIgbmVlZHMgdGhlIFNlbnRlbmNlUGllY2UgdG9rZW5pemVyLm1vZGVsIG5leHQgdG8gdGhlIHdlaWdodHMKICAgIHNwX3BhdGggPSBvcy5wYXRoLmpvaW4obWVyZ2VkX2RpciwgInRva2VuaXplci5tb2RlbCIpCiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMoc3BfcGF0aCk6CiAgICAgICAgZnJvbSBodWdnaW5nZmFjZV9odWIgaW1wb3J0IGhmX2h1Yl9kb3dubG9hZAogICAgICAgIHNodXRpbC5jb3B5KGhmX2h1Yl9kb3dubG9hZChhcmdzLm1vZGVsX2lkLCAidG9rZW5pemVyLm1vZGVsIiksIHNwX3BhdGgpCiAgICBwcmludChmIm1lcmdlZCBmcDE2IG1vZGVsIHNhdmVkIC0+IHttZXJnZWRfZGlyfSIpCiAgICBwcmludCgiZmlsZXM6Iiwgc29ydGVkKG9zLmxpc3RkaXIobWVyZ2VkX2RpcikpKQoKCmRlZiBtYWluKGFyZ3Y9Tm9uZSk6CiAgICBhcmdzID0gcGFyc2VfYXJncyhhcmd2KQogICAgb3MubWFrZWRpcnMoYXJncy5vdXRwdXRfZGlyLCBleGlzdF9vaz1UcnVlKQogICAgaWYgbm90IGFyZ3MubWVyZ2Vfb25seToKICAgICAgICB0cmFpbihhcmdzKQogICAgaWYgbm90IGFyZ3Mubm9fbWVyZ2U6CiAgICAgICAgbWVyZ2UoYXJncykKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==',
    'convert_to_mediapipe.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJtZXJnZWRfZnAxNiAtPiBJTlQ0IFRGTGl0ZSAoYWktZWRnZS10b3JjaCkgLT4gTWVkaWFQaXBlIC50YXNrIGJ1bmRsZSAoZ3VpZGUgcGFnZSA4KS4KCiAgcHl0aG9uMyBjb252ZXJ0X3RvX21lZGlhcGlwZS5weSBcXAogICAgICAtLW1lcmdlZC1kaXIgb3V0cHV0cy9nZW1tYTItMmItYmFua3NhdGhpL21lcmdlZF9mcDE2IFxcCiAgICAgIC0tY29udGV4dC1sZW4gNTEyIC0tb3V0cHV0LXRhc2sgb3V0cHV0cy9nZW1tYTItMmItYmFua3NhdGhpLWNwdS1pbnQ0LnRhc2sKClJ1biBpbiBhIEZSRVNIIENvbGFiIHJ1bnRpbWUgYWZ0ZXIgYHBpcCBpbnN0YWxsIC1yIHJlcXVpcmVtZW50cy1jb252ZXJ0LnR4dGA7IGFpLWVkZ2UtdG9yY2ggcGlucyBpdHMgb3duCnRvcmNoIHZlcnNpb24gdGhhdCBjb25mbGljdHMgd2l0aCB0aGUgdHJhaW5pbmcgc3RhY2suIE5lZWRzIH4xMiBHQiBzeXN0ZW0gUkFNIChDb2xhYiBIaWdoLVJBTSBoZWxwcykuCgpUaGUgYWktZWRnZS10b3JjaCBnZW5lcmF0aXZlIEFQSSBjaGFuZ2VkIGJldHdlZW4gcmVsZWFzZXMsIHNvIHNpZ25hdHVyZXMgYXJlIGRldGVjdGVkIGF0IHJ1bnRpbWUuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGdsb2IKaW1wb3J0IGluc3BlY3QKaW1wb3J0IG9zCmltcG9ydCBzeXMKaW1wb3J0IHRlbXBmaWxlCgoKZGVmIGltcG9ydF9nZW5lcmF0aXZlKCk6CiAgICBmb3IgcHJlZml4IGluICgiYWlfZWRnZV90b3JjaC5nZW5lcmF0aXZlIiwgImxpdGVydF90b3JjaC5nZW5lcmF0aXZlIik6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBnZW1tYTIgPSBfX2ltcG9ydF9fKGYie3ByZWZpeH0uZXhhbXBsZXMuZ2VtbWEiLCBmcm9tbGlzdD1bImdlbW1hMiJdKS5nZW1tYTIKICAgICAgICAgICAgY29udmVydGVyID0gX19pbXBvcnRfXyhmIntwcmVmaXh9LnV0aWxpdGllcyIsIGZyb21saXN0PVsiY29udmVydGVyIl0pLmNvbnZlcnRlcgogICAgICAgICAgICByZXR1cm4gZ2VtbWEyLCBjb252ZXJ0ZXIKICAgICAgICBleGNlcHQgKEltcG9ydEVycm9yLCBBdHRyaWJ1dGVFcnJvcik6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICBzeXMuZXhpdCgiYWktZWRnZS10b3JjaCBpcyBub3QgaW5zdGFsbGVkOiBwaXAgaW5zdGFsbCAtciByZXF1aXJlbWVudHMtY29udmVydC50eHQiKQoKCmRlZiBidWlsZF9tb2RlbChnZW1tYTIsIG1lcmdlZF9kaXI6IHN0ciwgY29udGV4dF9sZW46IGludCk6CiAgICBwYXJhbXMgPSBpbnNwZWN0LnNpZ25hdHVyZShnZW1tYTIuYnVpbGRfMmJfbW9kZWwpLnBhcmFtZXRlcnMKICAgIGt3YXJncyA9IHsia3ZfY2FjaGVfbWF4X2xlbiI6IGNvbnRleHRfbGVufSBpZiAia3ZfY2FjaGVfbWF4X2xlbiIgaW4gcGFyYW1zIGVsc2Uge30KICAgIHJldHVybiBnZW1tYTIuYnVpbGRfMmJfbW9kZWwobWVyZ2VkX2RpciwgKiprd2FyZ3MpCgoKZGVmIHRvX3RmbGl0ZShjb252ZXJ0ZXIsIG1vZGVsLCB3b3JrX2Rpcjogc3RyLCBwcmVmaWxsOiBpbnQsIHF1YW50aXplOiBzdHIpIC0+IHN0cjoKICAgIGZuID0gY29udmVydGVyLmNvbnZlcnRfdG9fdGZsaXRlCiAgICBwYXJhbXMgPSBpbnNwZWN0LnNpZ25hdHVyZShmbikucGFyYW1ldGVycwogICAgaWYgIm91dHB1dF9uYW1lX3ByZWZpeCIgaW4gcGFyYW1zOiAgIyBhaS1lZGdlLXRvcmNoID49IDAuMwogICAgICAgIGV4dHJhID0geyJrdl9jYWNoZV9tYXhfbGVuIjogcHJlZmlsbH0gaWYgImt2X2NhY2hlX21heF9sZW4iIGluIHBhcmFtcyBlbHNlIHt9ICAjIHJlcXVpcmVkIGluIGxpdGVydC10b3JjaCA+PSAwLjEwCiAgICAgICAgZm4obW9kZWwsIG91dHB1dF9wYXRoPXdvcmtfZGlyLCBvdXRwdXRfbmFtZV9wcmVmaXg9ImdlbW1hMl8yYl9iYW5rc2F0aGkiLAogICAgICAgICAgIHByZWZpbGxfc2VxX2xlbj1wcmVmaWxsLCBxdWFudGl6ZT1xdWFudGl6ZSwgKipleHRyYSkKICAgIGVsc2U6ICAjIDAuMi54OiB0ZmxpdGVfcGF0aCArIGJvb2xlYW4gcXVhbnRpemUgKGR5bmFtaWMgaW50OCkKICAgICAgICBwcmludCgibm90ZTogb2xkIGFpLWVkZ2UtdG9yY2ggZGV0ZWN0ZWQ7IGl0IG9ubHkgc3VwcG9ydHMgZHluYW1pYyBJTlQ4LCBub3QgSU5UNCIpCiAgICAgICAgZm4obW9kZWwsIG9zLnBhdGguam9pbih3b3JrX2RpciwgImdlbW1hMl8yYl9iYW5rc2F0aGkudGZsaXRlIiksIHByZWZpbGxfc2VxX2xlbj1wcmVmaWxsLCBxdWFudGl6ZT1UcnVlKQogICAgZm91bmQgPSBzb3J0ZWQoZ2xvYi5nbG9iKG9zLnBhdGguam9pbih3b3JrX2RpciwgIioudGZsaXRlIikpLCBrZXk9b3MucGF0aC5nZXRzaXplLCByZXZlcnNlPVRydWUpCiAgICBpZiBub3QgZm91bmQ6CiAgICAgICAgc3lzLmV4aXQoImNvbnZlcnNpb24gcHJvZHVjZWQgbm8gLnRmbGl0ZSBmaWxlIikKICAgIHJldHVybiBmb3VuZFswXQoKCmRlZiBidW5kbGUodGZsaXRlX3BhdGg6IHN0ciwgdG9rZW5pemVyX3BhdGg6IHN0ciwgb3V0cHV0X3Rhc2s6IHN0ciwgZW1iZWRfdGVtcGxhdGU6IGJvb2wpOgogICAgZnJvbSBtZWRpYXBpcGUudGFza3MucHl0aG9uLmdlbmFpIGltcG9ydCBidW5kbGVyCgogICAgZmllbGRzID0gaW5zcGVjdC5zaWduYXR1cmUoYnVuZGxlci5CdW5kbGVDb25maWcpLnBhcmFtZXRlcnMKICAgIGNmZyA9IGRpY3QoCiAgICAgICAgdGZsaXRlX21vZGVsPXRmbGl0ZV9wYXRoLAogICAgICAgIHRva2VuaXplcl9tb2RlbD10b2tlbml6ZXJfcGF0aCwKICAgICAgICBzdGFydF90b2tlbj0iPGJvcz4iLAogICAgICAgIHN0b3BfdG9rZW5zPVsiPGVvcz4iLCAiPGVuZF9vZl90dXJuPiJdLAogICAgICAgIG91dHB1dF9maWxlbmFtZT1vdXRwdXRfdGFzaywKICAgICAgICBlbmFibGVfYnl0ZXNfdG9fdW5pY29kZV9tYXBwaW5nPUZhbHNlLAogICAgKQogICAgaWYgZW1iZWRfdGVtcGxhdGUgYW5kICJwcm9tcHRfcHJlZml4IiBpbiBmaWVsZHM6CiAgICAgICAgIyBPbmx5IGlmIEh5YnJpZEFpU2VydmljZS5rdCBzZW5kcyB0aGUgcmF3IHByb21wdCBXSVRIT1VUIEdlbW1hIHR1cm4gbWFya2Vycy4KICAgICAgICBjZmdbInByb21wdF9wcmVmaXgiXSA9ICI8c3RhcnRfb2ZfdHVybj51c2VyXG4iCiAgICAgICAgY2ZnWyJwcm9tcHRfc3VmZml4Il0gPSAiPGVuZF9vZl90dXJuPlxuPHN0YXJ0X29mX3R1cm4+bW9kZWxcbiIKICAgIGJ1bmRsZXIuY3JlYXRlX2J1bmRsZShidW5kbGVyLkJ1bmRsZUNvbmZpZygqKmNmZykpCgoKZGVmIG1haW4oYXJndj1Ob25lKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXywgZm9ybWF0dGVyX2NsYXNzPWFyZ3BhcnNlLlJhd0Rlc2NyaXB0aW9uSGVscEZvcm1hdHRlcikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1tZXJnZWQtZGlyIiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1jb250ZXh0LWxlbiIsIHR5cGU9aW50LCBkZWZhdWx0PTUxMiwgaGVscD0icHJlZmlsbCArIEtWIGNhY2hlIGxlbmd0aCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0cHV0LXRhc2siLCBkZWZhdWx0PSJvdXRwdXRzL2dlbW1hMi0yYi1iYW5rc2F0aGktY3B1LWludDQudGFzayIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcXVhbnRpemUiLCBkZWZhdWx0PSJkeW5hbWljX2ludDRfYmxvY2szMiIsCiAgICAgICAgICAgICAgICAgICAgaGVscD0iYWktZWRnZS10b3JjaCByZWNpcGU6IGR5bmFtaWNfaW50NF9ibG9jazMyIHwgZHluYW1pY19pbnQ0X2Jsb2NrMTI4IHwgZHluYW1pY19pbnQ4IikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1lbWJlZC1jaGF0LXRlbXBsYXRlIiwgYWN0aW9uPSJzdG9yZV90cnVlIiwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJzdG9yZSBHZW1tYSB0dXJuIG1hcmtlcnMgaW4gdGhlIGJ1bmRsZSAodXNlIG9ubHkgaWYgdGhlIGFwcCBkb2VzIG5vdCBhZGQgdGhlbSkiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWtlZXAtdGZsaXRlIiwgaGVscD0iYWxzbyBjb3B5IHRoZSBpbnRlcm1lZGlhdGUgLnRmbGl0ZSBoZXJlIikKICAgIGFyZ3MgPSBhcC5wYXJzZV9hcmdzKGFyZ3YpCgogICAgdG9rZW5pemVyX3BhdGggPSBvcy5wYXRoLmpvaW4oYXJncy5tZXJnZWRfZGlyLCAidG9rZW5pemVyLm1vZGVsIikKICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyh0b2tlbml6ZXJfcGF0aCk6CiAgICAgICAgc3lzLmV4aXQoZiJtaXNzaW5nIHt0b2tlbml6ZXJfcGF0aH0gKHJlLXJ1biB0cmFpbl9nZW1tYTJiX3Fsb3JhLnB5IC0tbWVyZ2Utb25seSkiKQogICAgb3MubWFrZWRpcnMob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChhcmdzLm91dHB1dF90YXNrKSksIGV4aXN0X29rPVRydWUpCgogICAgZ2VtbWEyLCBjb252ZXJ0ZXIgPSBpbXBvcnRfZ2VuZXJhdGl2ZSgpCiAgICBwcmludCgiYnVpbGRpbmcgYWktZWRnZS10b3JjaCBHZW1tYS0yIDJCIGdyYXBoIGZyb20iLCBhcmdzLm1lcmdlZF9kaXIpCiAgICBtb2RlbCA9IGJ1aWxkX21vZGVsKGdlbW1hMiwgYXJncy5tZXJnZWRfZGlyLCBhcmdzLmNvbnRleHRfbGVuKQogICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoKSBhcyB3b3JrOgogICAgICAgIHByaW50KGYiY29udmVydGluZyAtPiBURkxpdGUgKHthcmdzLnF1YW50aXplfSwgcHJlZmlsbD17YXJncy5jb250ZXh0X2xlbn0sIGt2PXthcmdzLmNvbnRleHRfbGVufSkgLi4uIikKICAgICAgICB0ZmxpdGUgPSB0b190ZmxpdGUoY29udmVydGVyLCBtb2RlbCwgd29yaywgYXJncy5jb250ZXh0X2xlbiwgYXJncy5xdWFudGl6ZSkKICAgICAgICBwcmludChmInRmbGl0ZToge29zLnBhdGguZ2V0c2l6ZSh0ZmxpdGUpIC8gMTAyNCAqKiAzOi4yZn0gR0IiKQogICAgICAgIGlmIGFyZ3Mua2VlcF90ZmxpdGU6CiAgICAgICAgICAgIGltcG9ydCBzaHV0aWwKICAgICAgICAgICAgc2h1dGlsLmNvcHkodGZsaXRlLCBhcmdzLmtlZXBfdGZsaXRlKQogICAgICAgIHByaW50KCJidW5kbGluZyB3aXRoIE1lZGlhUGlwZSAuLi4iKQogICAgICAgIGJ1bmRsZSh0ZmxpdGUsIHRva2VuaXplcl9wYXRoLCBhcmdzLm91dHB1dF90YXNrLCBhcmdzLmVtYmVkX2NoYXRfdGVtcGxhdGUpCiAgICBwcmludChmImRvbmUgLT4ge2FyZ3Mub3V0cHV0X3Rhc2t9ICh7b3MucGF0aC5nZXRzaXplKGFyZ3Mub3V0cHV0X3Rhc2spIC8gMTAyNCAqKiAzOi4yZn0gR0IpIikKICAgIHByaW50KCJhZGIgcHVzaCIsIGFyZ3Mub3V0cHV0X3Rhc2ssICIvZGF0YS9sb2NhbC90bXAvbGxtL2dlbW1hMi0yYi1pdC1jcHUtaW50NC50YXNrIikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==',
    'requirements.txt': 'IyBUcmFpbmluZyArIGV2YWx1YXRpb24gc3RhY2sgKEdvb2dsZSBDb2xhYiBUNCkuIHRvcmNoIGNvbWVzIHByZWluc3RhbGxlZCBvbiBDb2xhYi4KdHJhbnNmb3JtZXJzPj00LjQ0LjAKcGVmdD49MC4xMi4wCmFjY2VsZXJhdGU+PTAuMzMuMApiaXRzYW5kYnl0ZXM+PTAuNDMuMwpkYXRhc2V0cz49Mi4yMC4wCnNlbnRlbmNlcGllY2U+PTAuMi4wCmh1Z2dpbmdmYWNlX2h1Yj49MC4yNC4wCg==',
    'requirements-convert.txt': 'IyBJTlQ0IFRGTGl0ZSBjb252ZXJzaW9uICsgTWVkaWFQaXBlIGJ1bmRsaW5nLiBJbnN0YWxsIGluIGEgRlJFU0ggUHl0aG9uIHNlc3Npb246CiMgbGl0ZXJ0LXRvcmNoIChmb3JtZXJseSBhaS1lZGdlLXRvcmNoKSBwaW5zIGl0cyBvd24gdG9yY2ggYnVpbGQgYW5kIGNvbmZsaWN0cyB3aXRoIHRoZSB0cmFpbmluZyBzdGFjay4KbGl0ZXJ0LXRvcmNoPj0wLjEwLjAKIyBtZWRpYXBpcGUgPj0gMS4wIGRyb3BwZWQgbWVkaWFwaXBlLnRhc2tzLnB5dGhvbi5nZW5haSAodGhlIC50YXNrIGJ1bmRsZXIpOyAwLjEwLjM1IGhhcyBpdCBhbmQgc3VwcG9ydHMgUHl0aG9uIDMuMTMKbWVkaWFwaXBlPT0wLjEwLjM1CnNhZmV0ZW5zb3JzCnNlbnRlbmNlcGllY2UK',
    'README.md': 'IyBCYW5rU2F0aGkgwrcgR2VtbWEtMkIgUUxvUkEgdHJhaW5pbmcgcGlwZWxpbmUKClRoZSBjb2RlIGZvciB0aGUgKkdlbW1hLTJCIFFMb1JBIFRyYWluaW5nICYgMWstM2sgRGF0YXNldCBFbmdpbmVlcmluZyBHdWlkZSouIEl0IGZpbmUtdHVuZXMKYGdvb2dsZS9nZW1tYS0yLTJiLWl0YCBvbiBhIGZyZWUgQ29sYWIgVDQgc28gaXQgZml0cyB0aGUgdHdvIGBIeWJyaWRBaVNlcnZpY2Uua3RgIEpTT04gY29udHJhY3RzLiBUaGVuIGl0CmNvbnZlcnRzIHRoZSByZXN1bHQgdG8gYW4gSU5UNCBNZWRpYVBpcGUgYC50YXNrYCBidW5kbGUuCgp8IEZpbGUgfCBHdWlkZSBzdGFnZSB8IFdoYXQgaXQgZG9lcyB8CnwtLS18LS0tfC0tLXwKfCBgYmFua3NhdGhpX2NvbW1vbi5weWAgfCBzaGFyZWQgfCBQcm9tcHQgaGVhZGVycywgSlNPTiBzY2hlbWFzLCBHZW1tYS0yIGNoYXQgd3JhcHBlciwgU0hBLTI1NiBkZWR1cGUsIERldmFuYWdhcmkgY2hlY2sgfAp8IGBkYXRhc2V0X2dlbmVyYXRvci5weWAgfCBTdGFnZSAyIHwgU3RyYXRpZmllZCAxay0zayBkYXRhc2V0LCAzIHJlZ2lzdGVycyArIEFTUiBub2lzZSwgUkFHIGdyb3VuZGluZywgYWJzdGVudGlvbiB0cmFwcywgU2NhbUd1YXJkIHRpZXJzIHwKfCBgZXZhbF9nZW1tYV9iYW5rc2F0aGkucHlgIHwgU3RhZ2UgNCB8IFByZS1HUFUgZGF0YXNldCBhdWRpdCAqKmFuZCoqIGxpdmUgNS1LUEkgaG9sZC1vdXQgZ2F0ZSAob3Igc2NvcmVzIHNhdmVkIG91dHB1dHMpIHwKfCBgdHJhaW5fZ2VtbWEyYl9xbG9yYS5weWAgfCBTdGFnZSAzIHwgNC1iaXQgTkY0IFFMb1JBLCBjb21wbGV0aW9uLW9ubHkgbG9zcyBtYXNraW5nLCBgbWVyZ2VfYW5kX3VubG9hZCgpYCB0byBgbWVyZ2VkX2ZwMTYvYCB8CnwgYGNvbnZlcnRfdG9fbWVkaWFwaXBlLnB5YCB8IFN0YWdlIDQgfCBhaS1lZGdlLXRvcmNoIElOVDQgVEZMaXRlLCBNZWRpYVBpcGUgYnVuZGxlciwgYGJhbmtzYXRoaS1jcHUtaW50NC50YXNrYCB8CnwgYEJhbmtTYXRoaV9HZW1tYTJCX0NvbGFiLmlweW5iYCB8IGFsbCB8IENsaWNrLXRocm91Z2ggQ29sYWIgcnVuYm9vayB8CgojIyBSdW4gaXQgb24gQ29sYWIKCjEuIFVwbG9hZCBgQmFua1NhdGhpX0dlbW1hMkJfQ29sYWIuaXB5bmJgIHRvIGh0dHBzOi8vY29sYWIucmVzZWFyY2guZ29vZ2xlLmNvbSBhbmQgY2hvb3NlIGEgKipUNCBHUFUqKiBydW50aW1lLgoyLiBSdW4gdGhlIGNlbGxzIHRvcCB0byBib3R0b20uIFdoZW4gYXNrZWQsIHVwbG9hZCBgZ2VtbWFfdHJhaW5pbmcuemlwYCwgd2hpY2ggc2l0cyBuZXh0IHRvIHRoaXMgZm9sZGVyLgozLiBQYXN0ZSBhIEh1Z2dpbmcgRmFjZSB0b2tlbi4gWW91IG11c3QgYWNjZXB0IHRoZSBgZ29vZ2xlL2dlbW1hLTItMmItaXRgIGxpY2VuY2UgZmlyc3QuCgpZb3UgY2FuIGFsc28gcnVuIGl0IGZyb20gYSB0ZXJtaW5hbCAoZ3VpZGUgcC43KToKCmBgYGJhc2gKcGlwIGluc3RhbGwgLXEgLXIgcmVxdWlyZW1lbnRzLnR4dCAmJiBodWdnaW5nZmFjZS1jbGkgbG9naW4KcHl0aG9uMyBkYXRhc2V0X2dlbmVyYXRvci5weSAtLW91dC1kaXIgZGF0YSAtLXRhcmdldC1yb3dzIDE1MDAKcHl0aG9uMyBldmFsX2dlbW1hX2JhbmtzYXRoaS5weSAtLXRyYWluLWZpbGUgZGF0YS90cmFpbi5qc29ubCAtLWV2YWwtZmlsZSBkYXRhL2V2YWwuanNvbmwgICAjIG11c3QgUEFTUwpweXRob24zIHRyYWluX2dlbW1hMmJfcWxvcmEucHkgLS1tb2RlbC1pZCBnb29nbGUvZ2VtbWEtMi0yYi1pdCAtLXRyYWluLWZpbGUgZGF0YS90cmFpbi5qc29ubCBcCiAgICAtLWV2YWwtZmlsZSBkYXRhL2V2YWwuanNvbmwgLS1lcG9jaHMgNCAtLW91dHB1dC1kaXIgb3V0cHV0cy9nZW1tYTItMmItYmFua3NhdGhpCnB5dGhvbjMgZXZhbF9nZW1tYV9iYW5rc2F0aGkucHkgLS1tb2RlbC1kaXIgb3V0cHV0cy9nZW1tYTItMmItYmFua3NhdGhpL21lcmdlZF9mcDE2IC0tZXZhbC1maWxlIGRhdGEvZXZhbC5qc29ubAojIGZyZXNoIHJ1bnRpbWU6CnBpcCBpbnN0YWxsIC1yIHJlcXVpcmVtZW50cy1jb252ZXJ0LnR4dApweXRob24zIGNvbnZlcnRfdG9fbWVkaWFwaXBlLnB5IC0tbWVyZ2VkLWRpciBvdXRwdXRzL2dlbW1hMi0yYi1iYW5rc2F0aGkvbWVyZ2VkX2ZwMTYgXAogICAgLS1jb250ZXh0LWxlbiA1MTIgLS1vdXRwdXQtdGFzayBvdXRwdXRzL2dlbW1hMi0yYi1iYW5rc2F0aGktY3B1LWludDQudGFzawpgYGAKCiMjIERhdGFzZXQgZGVzaWduIChndWlkZSBwcC4gMy01KQoKKiAqKlF1b3RhczoqKiB0aGUgd2VpZ2h0cyBhcmUgbG9hbl9rY2MgLjE0LCBwYXNzYm9va19kYnRfYnNiZGEgLjE0LCB0cmFuc2Zlcl9pcHBiX2JjIC4xNCwgc2FmZV9hYnN0ZW50aW9uIC4xNiwKICB3aXRoZHJhd19iY19sZV8xMGsgLjEyLCB3aXRoZHJhd19icmFuY2hfZ3RfMTBrIC4xMCwgYXNzaXN0YW50X3NjYW1fYWxlcnQgLjEwLCBzY2FtX2hpZ2hfcmlzayAuMDQsCiAgc2NhbV9jYXV0aW9uIC4wMyBhbmQgc2NhbV9zYWZlIC4wMy4gWW91IGNhbiBjaGFuZ2UgdGhlbSBpbiBgQlVDS0VUX1dFSUdIVFNgLgoqICoqUmVnaXN0ZXJzOioqIGFib3V0IDM4JSBFbmdsaXNoLCAyNCUgRGV2YW5hZ2FyaSBIaW5kaSBhbmQgMzglIEhpbmdsaXNoLiBUaGUgSGluZ2xpc2ggcm93cyBnZXQgc2ltdWxhdGVkCiAgV2hpc3Blci10aW55IHNsaXBzOiBsb3dlcmNhc2UgdGV4dCwgYSBkcm9wcGVkIGA/YCwgYGFhZGhhcmAsIGBwYXNidWtgLCBgcnVwZWVzYCwgZmlsbGVycyBhbmQgc3R1dHRlcnMuIFRoZQogIHRhcmdldHMgYWx3YXlzIHN0YXkgYXMgY2xlYW4gRW5nbGlzaCBwbHVzIERldmFuYWdhcmkuCiogKipCb3VuZGFyaWVzOioqIGFtb3VudHMgb2YgUnMgOSw5OTksIDEwLDAwMCBhbmQgMTAsMDAxIHRlc3QgQkMgdmVyc3VzIEJyYW5jaC9BVE0gcm91dGluZy4gRm9yIEtDQywgYW1vdW50cyBvZgogIFJzIDEsOTksOTk5LCAyLDAwLDAwMCwgMiwwMCwwMDEgYW5kIDIsNTAsMDAwIHRlc3QgdGhlIGNvbGxhdGVyYWwtZnJlZSBsaW1pdC4gV2l0aGRyYXdhbHMgYXQgb3IgYmVsb3cgUnMgMTBrIGFuZAogIGFib3ZlIFJzIDEwayB1c2UgdGhlICpzYW1lKiBxdWVyeSB0ZW1wbGF0ZXMsIHNvIHRoZSBtb2RlbCBsZWFybnMgdGhlIHRocmVzaG9sZCByYXRoZXIgdGhhbiB0aGUgd29yZGluZy4KKiAqKkdyb3VuZGluZzoqKiBhYm91dCA1MCUgb2YgVGFzayBBIHByb21wdHMgY2FycnkgYSBgW1NPVVJDRV9JRCB8IFZlcmlmaWVkIDIwMjYtMTEtMTRdYCBzbmlwcGV0LiAzMCUgb2YgdGhvc2UKICBhbHNvIGNhcnJ5IGEgZGlzdHJhY3RvciBzbmlwcGV0LiBBYnN0ZW50aW9uIHRyYXBzIGNhbiBjb21lIHdpdGggYW4gb24tdG9waWMgc25pcHBldCB0aGF0IHN0aWxsIGRvZXMgbm90CiAgYW5zd2VyIHRoZSBxdWVzdGlvbi4KKiAqKkdhdGVzOioqIHF1ZXJpZXMgYXJlIGRlZHVwbGljYXRlZCBieSBTSEEtMjU2IGFmdGVyIHJlbW92aW5nIHB1bmN0dWF0aW9uIGFuZCBjYXNlLCBhbmQgdGhlIHNwbGl0IGlzIGEKICBzdHJhdGlmaWVkIDg1LzE1LiBMZWFrYWdlIGlzIGNoZWNrZWQgYW5kIG11c3QgYmUgMC4gRXZlcnkgdGFyZ2V0IGlzIGNoZWNrZWQgYWdhaW5zdCB0aGUgc2NoZW1hLCB0aGUKICA0NS13b3JkIGxpbWl0ICg1MCBmb3IgVGFzayBCKSBhbmQgRGV2YW5hZ2FyaSBwYXJpdHkuCgpFYWNoIEpTT05MIHJvdyBjb250YWlucyBgdGV4dGAsIHdoaWNoIGlzIHRoZSBHZW1tYS0yIGNoYXQgdHVybiB1c2VkIGZvciB0cmFpbmluZy4gSXQgYWxzbyBjb250YWlucwpgcHJvbXB0YCwgYGNvbXBsZXRpb25gLCBgYnVja2V0YCwgYHJlZ2lzdGVyYCwgYGdyb3VuZGVkYCwgYGNvbnRleHRfaWRzYCwgYHF1ZXJ5YCBhbmQgYHF1ZXJ5X3NoYTI1NmAuCgojIyBQcm9tcHQgY29udHJhY3QKClRoZSBwcm9tcHQgaGVhZGVycyBpbiBgYmFua3NhdGhpX2NvbW1vbi5weWAgKGBUQVNLX0FfSEVBREVSYCwgYFRBU0tfQl9IRUFERVJgLCBgYnVpbGRfdGFza19hX3Byb21wdGApIG11c3QgbWF0Y2gKd2hhdCBgSHlicmlkQWlTZXJ2aWNlLmt0YCBzZW5kcyAqKmV4YWN0bHkqKjogdGhlIHNhbWUgaGVhZGVyLCB0aGUgYFZlcmlmaWVkIENvbnRleHQ6YCBibG9jaywgdGhlbiBgVXNlcjogPHF1ZXJ5PmAuCklmIHRoZSBhcHAncyB3b3JkaW5nIGRpZmZlcnMsIGNoYW5nZSBpdCBpbiB0aGF0IG9uZSBmaWxlIGFuZCByZWdlbmVyYXRlIHRoZSBkYXRhc2V0LiBPdGhlcndpc2UgdGhlIG1vZGVsIGlzCnNlcnZlZCBwcm9tcHRzIGl0IHdhcyBuZXZlciB0cmFpbmVkIG9uLiBUaGUgYXBwIHNob3VsZCB3cmFwIHByb21wdHMgYXMKYDxzdGFydF9vZl90dXJuPnVzZXJcbntwcm9tcHR9PGVuZF9vZl90dXJuPlxuPHN0YXJ0X29mX3R1cm4+bW9kZWxcbmAsIHVubGVzcyB5b3UgYnVuZGxlIHdpdGgKYC0tZW1iZWQtY2hhdC10ZW1wbGF0ZWAuCgojIyBGYWlsdXJlIHBsYXlib29rIChndWlkZSBwLjcpCgp8IFN5bXB0b20gfCBGaXggfAp8LS0tfC0tLXwKfCBgdmFsaWRfanNvbl9wY3RgIDwgOTkgfCBDaGVjayB0aGF0IHRoZSBgW21hc2sgY2hlY2tdYCBsaW5lIHByaW50ZWQgZHVyaW5nIHRyYWluaW5nIHN0YXJ0cyBhdCBge2AuIFRyeSBgLS1lcG9jaHMgNWAuIHwKfCBXSVRIRFJBVyBwcmVkaWN0ZWQgZm9yIHRyYW5zZmVyL0RCVCBxdWVyaWVzIHwgS2VlcCB3aXRoZHJhd18qIGF0IG9yIGJlbG93IDI1JSAodGhlIGF1ZGl0IGVuZm9yY2VzIHRoaXMpLiBSYWlzZSB0aGUgdHJhbnNmZXIvcGFzc2Jvb2sgd2VpZ2h0cy4gfAp8IGBzY2FtZ3VhcmRfcmVjYWxsX3BjdGAgPCAxMDAgfCBBZGQgdGhlIG1pc3NlZCBwaHJhc2luZyB0byBgU0NBTV9BTEVSVF9RYCAvIGBISUdIX1JJU0tfTVNHU2AuIEtlZXAgdGhlIFJvb20gc2V2ZXJpdHkgcnVsZXMuIHwKfCBgc2FmZV9hYnN0ZW50aW9uX3BjdGAgPCA5NSB8IFJhaXNlIGBzYWZlX2Fic3RlbnRpb25gIGZyb20gMC4xNiB0byAwLjIyLiBBZGQgbW9yZSB0ZW1wbGF0ZXMgdG8gYEFCU1RBSU5fUWAuIHwKfCBgcmVwbHlIaWAgbG9zZXMgRGV2YW5hZ2FyaSBvciBydW5zIHBhc3QgNDUgd29yZHMgfCBTaG9ydGVuIHRoZSByZXBseSB0ZW1wbGF0ZXMuIFRoZSBhdWRpdCdzIGBkZXZhbmFnYXJpX29rYCB3aWxsIGZsYWcgaXQuIHwKCkZhaWxlZCByb3dzIGFyZSBwcmludGVkIHVuZGVyIGBmYWlsdXJlc19zYW1wbGVgLCBhbmQgYC0tc2F2ZS1wcmVkaWN0aW9uc2Aga2VlcHMgZXZlcnkgcmF3IG91dHB1dC4KCiMjIEFuZHJvaWQgc21va2UgdGVzdCAoYWlycGxhbmUgbW9kZSwgZ3VpZGUgcC44KQoKMS4gYEJoYWl5YSA1MDAwIHJ1cGF5ZSBuaWthbG5lIGhhaW5gIOKGkiBXSVRIRFJBVywgNTAwMCwgQmFuayBNaXRyYSwgbm8gZmVlCjIuIGAyNTAwMCBydXBlZXMgbmlrYWxuZSBoYWluYCDihpIgV0lUSERSQVcsIDI1MDAwLCBCcmFuY2gvQVRNCjMuIGAxNTAwMDAgS0NDIGxvYW4gZG9jdW1lbnRzP2Ag4oaSIExPQU5fS0NDLCBScyAyTCBjb2xsYXRlcmFsLWZyZWUsIEtoYXRhdW5pICsgMiBjb3BpZXMgKyAzIHBob3Rvcwo0LiBgQ2FsbGVyIGFza2VkIE9UUCAmIEFueURlc2tgIOKGkiBTY2FtR3VhcmQgZmlyZXMgYmVmb3JlIHRoZSBMTE0sIGlzU2NhbUFsZXJ0PXRydWUsIDE5MzAgKyAzIGRheXMKNS4gYEV4YWN0IGJhbGFuY2UgaW4gU0JJIGFjY3QgMzA0OTE/YCDihpIgYWJzdGFpbmVkPXRydWUsIHRhc2sgTk9ORSwgQnJhbmNoIC8gMTQ0NDgKNi4gQSBQTS1LSVNBTiBjcmVkaXQgU01TIOKGkiByaXNrTGV2ZWwgU0FGRQoKS2VlcCBgQmFua2luZ1J1bGVFbmdpbmUuaXNJbW1lZGlhdGVTY2FtUGhyYXNlKClgIHJ1bm5pbmcgYmVmb3JlIGluZmVyZW5jZS4gS2VlcCB0aGUgZGV0ZXJtaW5pc3RpYyBmYWxsYmFjawpmb3Igd2hlbiBKU09OIHBhcnNpbmcgZmFpbHMuIEZpbmUtdHVuaW5nIGRvZXMgbm90IHJlcGxhY2UgZWl0aGVyIG9uZS4K',
}
os.makedirs('/content/gemma_training', exist_ok=True)
for name, b64 in FILES.items():
    with open(f'/content/gemma_training/{name}', 'wb') as fh:
        fh.write(base64.b64decode(b64))
%cd /content/gemma_training
!ls -la

In [ ]:
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao  # Colab's preinstalled torchao is too old for peft and breaks the LoRA merge

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # paste HF token (after accepting google/gemma-2-2b-it terms)

## Step 2 · Generate 1,500-row stratified dataset + pre-GPU audit (~5 s)
Must show `valid_json_pct: 100.0`, `devanagari_parity_pct: 100.0`, `train_eval_leakage_count: 0` and `RESULT: PASS`.

In [ ]:
!python3 dataset_generator.py --out-dir data --target-rows 1500
!python3 eval_gemma_banksathi.py --train-file data/train.jsonl --eval-file data/eval.jsonl --report-file data/audit_report.json

In [ ]:
import json
rows = [json.loads(l) for l in open('data/train.jsonl', encoding='utf-8')]
for b in ['withdraw_bc_le_10k', 'safe_abstention', 'assistant_scam_alert', 'scam_safe']:
    r = next(x for x in rows if x['bucket'] == b)
    print(f"--- {b} [{r['register']}] ---\nQ: {r['query']}\nA: {r['completion']}\n")

## Step 3 · QLoRA fine-tune + merge to FP16 (~22-30 min on T4)

In [ ]:
!python3 train_gemma2b_qlora.py --model-id google/gemma-2-2b-it \
    --train-file data/train.jsonl --eval-file data/eval.jsonl \
    --epochs 4 --output-dir outputs/gemma2-2b-banksathi

## Step 4 · Live 5-KPI hold-out safety benchmark (~3 min)
Gates: JSON ≥ 99 · intent ≥ 96 · ScamGuard recall = 100 · abstention ≥ 95 · bilingual TTS ≥ 98.
If a gate fails, see the failure playbook in README.md, adjust, and re-run Steps 2-4.

In [ ]:
!python3 eval_gemma_banksathi.py \
    --model-dir outputs/gemma2-2b-banksathi/merged_fp16 \
    --eval-file data/eval.jsonl \
    --save-predictions outputs/eval_predictions.jsonl \
    --report-file outputs/kpi_report.json

## Save to Google Drive (Colab disks are wiped when the runtime ends)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/banksathi
!cp -r data outputs/gemma2-2b-banksathi/adapter outputs/gemma2-2b-banksathi/train_metrics.json outputs/kpi_report.json /content/drive/MyDrive/banksathi/
!cp -r outputs/gemma2-2b-banksathi/merged_fp16 /content/drive/MyDrive/banksathi/
!ls -la /content/drive/MyDrive/banksathi

## Step 5 · INT4 MediaPipe `.task` conversion
`ai-edge-torch` pins its own torch build, so **do `Runtime → Restart session`** (or `Disconnect and delete runtime` for a clean one; a High-RAM CPU runtime is enough) and run only the cells below.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
assert os.path.exists('/content/gemma_training'), 'run the code cell in Step 1 first'
%cd /content/gemma_training
!pip install -q -r requirements-convert.txt

In [ ]:
!python3 convert_to_mediapipe.py \
    --merged-dir /content/drive/MyDrive/banksathi/merged_fp16 \
    --context-len 512 \
    --output-task /content/drive/MyDrive/banksathi/gemma2-2b-banksathi-cpu-int4.task

Download `gemma2-2b-banksathi-cpu-int4.task` (~1.45 GB) from Drive, then load it in BankSathi:

* **In-app:** Settings → *Import Gemma-2B (.task / .bin)*
* **ADB:** `adb push gemma2-2b-banksathi-cpu-int4.task /data/local/tmp/llm/gemma2-2b-it-cpu-int4.task`

Then run the 6-check airplane-mode smoke test in README.md.